## Contents

- [0 · Setup](#sec-0)
  - [0.D · Dataset overview table](#sec-0D)
  - [0.N · Aggregated numbers at hand](#sec-0N)
  - [0.P · Process discovery — load](#sec-0P)
  - [0.I · Individual profile realism — load](#sec-0I)
  - [0.C · Complete energy profile — load](#sec-0C)
- [1 · Results](#sec-1)
  - [1.P · Process discovery + timing](#sec-1P)
  - [1.I · Individual profile realism](#sec-1I)
  - [1.C · Complete energy profile](#sec-1C)
- [2 · LaTeX for the paper](#sec-2)
  - [2.P · Process table](#sec-2P)
  - [2.I · Individual profile realism table](#sec-2I)
  - [2.C · Complete energy profile table](#sec-2C)
- [3 · Evaluation counts](#sec-3)
  - [3.S · Summary — individual results per evaluation](#sec-3S)
  - [3.P · Process — processes (discovery) and cases (timing)](#sec-3P)
  - [3.I · Individual profile — one (process, activity, sensor) cell](#sec-3I)
  - [3.C · Complete profile — one (process, case, sensor) unit](#sec-3C)
- [4 · Full tables (anonymised)](#sec-4)
  - [4.P · Process — individual per-case results](#sec-4P)
  - [4.I · Individual profile — individual curves + breakdown ladder](#sec-4I)
  - [4.C · Complete profile — breakdown ladder](#sec-4C)


# Results

Merges `03_results_process`, `04_results_individual_profile` and
`05_results_complete_energy_profile`. Three evaluations, one notebook.

Each aggregates in **two stages**, so the unit named below is the unit of the
*last* one — the population section 3 counts:

| # | Evaluation | Stage 1 — one value per… | Stage 2 (the tables) — median over… | Direction |
|---|---|---|---|---|
| **P** timing | Process timing errors | **case** — read straight from `process_eval_per_case.parquet`, the raw per-case sample the pipeline's own medians were built from | all test **cases**, pooled (`Span WAPE` is a sum-over-sum across them) | **lower** |
| **P** discovery | Fitness / Precision / Generalization / Simplicity | **process** — model-level: the real log replayed against the net, per station, averaged. No per-case meaning exists | the **processes** | **higher** |
| **I** | Individual profile realism | **(process, activity, sensor) cell** — median over that cell's curves | those **cells** | **lower** (0 = perfect) |
| **C** | Complete energy profile | **(process, case, sensor)** — one curve against its own real counterpart; no stage-1 collapse, the case *is* the unit | those **units** | **lower** (0 = perfect) |

So the P timing columns and C are per case; the P discovery columns are the only
ones that cannot be, and I aggregates curves into cells first.

Because the timing columns pool cases, processes are **not** equally weighted —
one with more cases counts more. Section 3.P prints the share each carries.

Median at both stages: these errors are bounded below with a long right tail, so
a few bad cases or cells would drag any mean, and a mean of medians is neither
statistic.

**Nothing is reported per process.** The process is an identifier, not a result,
so it never appears as a row or column anywhere below.

Sections: **1** headline tables · **2** LaTeX for the paper · **3** evaluation
counts · **4** full breakdown tables (anonymised).

Names are prefixed `p_` / `i_` / `c_` (or `P_` / `I_` / `C_` for config) so the
three evaluations never collide.

<a id="sec-0"></a>

## 0 · Setup

In [1]:
# ── Shared config ────────────────────────────────────────────────────────────
import warnings, re
import numpy as np, pandas as pd
from pathlib import Path
from IPython.display import display, Markdown, HTML
warnings.filterwarnings('ignore')

RESULTS_ROOT = Path('..') / 'results'

# Identity labels (process / sensor / activity / case) become shuffled numbers.
# One seed for all three evaluations, so a label means the same thing everywhere
# the underlying label sets agree.
ANONYMISE = True
ANON_SEED = 20260730

SAVE_LATEX = True

# Experiments. P reads a different run from I/C on purpose — keep them separate.
EXPERIMENT = 1

P_SPLIT = 'test'      # split reported by the process tables
P_AGG   = 'median'    # aggregation across processes
I_SPLIT = 'TEST'      # 'TEST' | 'TRAIN'

# Model selection ALWAYS happens on TRAIN — selecting on the reported split would
# let a model be chosen for fitting the evaluation data.
SELECTION_SPLIT            = 'train'
SELECTION_METRIC_BASES     = ['conformance_metrics_fitness',
                              'conformance_metrics_precision',
                              'conformance_metrics_generalization',
                              'conformance_metrics_simplicity']
SELECTION_HIGHER_IS_BETTER = True     # quality scores: best = argmax


def _anon_map(values, prefix, seed):
    """Sorted real labels -> '<prefix> <n>', n a fixed random permutation of 1..N."""
    labels = sorted(pd.Series(values).dropna().astype(str).unique())
    order  = np.random.default_rng(seed).permutation(len(labels)) + 1
    return {lab: f'{prefix} {n}' for lab, n in zip(labels, order)}


def _tex_esc(s):
    return (str(s).replace('\\', r'\textbackslash ').replace('&', r'\&')
                  .replace('_', r'\_').replace('%', r'\%').replace('#', r'\#'))


def _latest_run(experiment):
    """Newest run dir for this experiment, by its TIMESTAMP.

    The name must be exactly 'experiment_<n>_<YYYYMMDD>_<HHMMSS>'. A plain
    startswith + sorted() picked up the retired 'experiment_1_old_...' and
    'experiment_1_failed_...' dirs and, because 'o'/'f' sort after the digit
    that starts a timestamp, returned the OLD run as the newest one.
    """
    pat  = re.compile(rf'^experiment_{experiment}_(\d{{8}}_\d{{6}})$')
    runs = sorted((m.group(1), d) for d in RESULTS_ROOT.iterdir()
                  if d.is_dir() and (m := pat.match(d.name)))
    assert runs, f'No runs for experiment {experiment}'
    return runs[-1][1]


def _parse_mode(m):
    """'petri_net_<model>[_ml_plus_*]' -> (model, time-prediction variant)."""
    r = str(m)
    if not r.startswith('petri_net_'):
        return r, 'baseline'
    r = r[len('petri_net_'):]
    if r.endswith('_ml_plus_global'):  return r[:-len('_ml_plus_global')],  'ml_global'
    if r.endswith('_ml_plus_per_act'): return r[:-len('_ml_plus_per_act')], 'ml_local'
    return r, 'baseline'


def _select_best_miner(df):
    """Combined-best per process: the pipeline's recorded TRAIN verdict, else the
    identical rule re-derived here. 'combined' is that verdict, not a miner, so it
    never competes against the miners it was selected from."""
    pre  = SELECTION_SPLIT.lower() + '_'
    cols = [pre + b for b in SELECTION_METRIC_BASES if (pre + b) in df.columns]
    assert cols, f'no {pre}* selection columns found'
    cands = sorted(set(df['model'].unique()) - {'alpha', 'budget', 'combined'})
    choice = {}
    if 'selected_mining_algorithm' in df.columns:
        cmb = df[(df['model'] == 'combined') & df['selected_mining_algorithm'].notna()]
        choice = cmb.groupby('process')['selected_mining_algorithm'].first().to_dict()
    best = {}
    for proc, g in df.groupby('process'):
        if proc in choice:
            best[proc] = choice[proc]; continue
        gc = g[g['model'].isin(cands)]
        if gc.empty:
            best[proc] = None; continue
        score = gc.groupby('model')[cols].mean().mean(axis=1)
        best[proc] = score.idxmax() if SELECTION_HIGHER_IS_BETTER else score.idxmin()
    src = 'pipeline, on TRAIN' if choice else f'notebook fallback, on {SELECTION_SPLIT.upper()}'
    print(f'Combined-best by {src} | candidates: {cands}')
    for p, m in best.items():
        print(f'  {p}: {m}')
    return best, cands

<a id="sec-0D"></a>

### 0.D · Dataset overview table

Table `table:datasets` in the paper. Cases and sensors come from the loaded
experiment run (`process_eval_per_case` / `real_test_curves`), which always
covers all six processes. Activities, **hours of operation** (recorded length
of the energy time series: unique timestamps × median sampling step, gaps
excluded) and **span (days)** (calendar period covered) need the raw gold
data and are filled per process only where `data/gold/experiment_1/<process>`
is present — on a machine without the full gold data those columns show `--`.

In [2]:
# ── D: dataset overview table ────────────────────────────────────────────────
# Cases + sensors from the experiment results (complete for all 6 processes);
# activities, hours of operation and span from gold data where available.
_d_run = _latest_run(EXPERIMENT)
print(f'run: {_d_run.name}')

_d_pc = pd.read_parquet(_d_run / 'process_eval_per_case.parquet')
_d_rc = pd.read_parquet(_d_run / 'real_test_curves.parquet')
d_datasets = pd.DataFrame({
    'number of cases':   _d_pc.groupby('process')['case_id'].nunique(),
    'number of sensors': _d_rc.groupby('Process')['Sensor'].nunique(),
}).rename_axis('dataset').reset_index()

_gold = Path('..') / 'data' / 'gold' / f'experiment_{EXPERIMENT}'
def _gold_stats(proc, n_cases_expected):
    _f = _gold / proc / 'datasets' / 'df_expanded.parquet'
    if not _f.exists():
        return None
    _df = pd.read_parquet(_f)
    # A gold folder is only trusted if its cases match the results run —
    # a stale snapshot can carry a same-named folder with different content.
    _n = _df['case_id_log'].nunique() if 'case_id_log' in _df.columns else None
    if _n != n_cases_expected:
        return None
    _ts = (pd.to_datetime(_df['datetime_energy'], errors='coerce')
             .dropna().drop_duplicates().sort_values())
    _step = _ts.diff().median()
    # Activities from the EVENT LOG, not the expanded table — activities that
    # never overlap the energy series are dropped from expanded and would be
    # undercounted there.
    _el = pd.read_parquet(_gold / proc / 'datasets' / 'df_event_log.parquet')
    return {
        'number of activities': _el['activity'].nunique(),
        'hours of operation': int(round(len(_ts) * _step.total_seconds() / 3600)),
        'span (days)': int(round((_ts.max() - _ts.min()).total_seconds() / 86400)),
    }

_g = {p: _gold_stats(p, n) for p, n in zip(d_datasets['dataset'], d_datasets['number of cases'])}
for _col in ('number of activities', 'hours of operation', 'span (days)'):
    d_datasets[_col] = [(_g[p] or {}).get(_col) for p in d_datasets['dataset']]
d_datasets = d_datasets[['dataset', 'number of cases', 'number of activities',
                         'number of sensors', 'hours of operation', 'span (days)']]

_missing = [p for p, v in _g.items() if v is None]
if _missing:
    print(f'WARNING: no gold data matching this run for {_missing} — '
          'activities/hours/span left empty for them; run on the machine with '
          'the full gold data to fill the table. Cases and sensors are '
          'complete (from the results run).\n')
display(d_datasets)
print(f"totals: {d_datasets['number of cases'].sum()} cases, "
      f"{d_datasets['number of sensors'].sum()} sensors, "
      f"{d_datasets['hours of operation'].sum(skipna=True):.0f} hours\n")

def _d_fmt(v):
    return '--' if pd.isna(v) else f'{int(v)}'

_lines = [
    r'\begin{table}[width=.9\linewidth,cols=6,pos=h]',
    r'\caption{Dataset information overview. Hours of operation refer to the recorded'
    r' length of the energy time series; span is the calendar period covered.}',
    r'\label{table:datasets}',
    r'\setlength{\tabcolsep}{3pt}',
    r'\begin{tabular}{@{}lrrrrr@{}}',
    r'\toprule',
    r'dataset & \makecell{number\\of cases} & \makecell{number of\\different\\activities}'
    r' & \makecell{number\\of sensors} & \makecell{hours of\\operation} & \makecell{span\\(days)} \\',
    r'\midrule',
]
for _, _r in d_datasets.iterrows():
    _name = _r['dataset'].replace('process_4_', 'process_4.').replace('_', r'\_')
    _lines.append(f"{_name} & {_d_fmt(_r['number of cases'])} & {_d_fmt(_r['number of activities'])}"
                  f" & {_d_fmt(_r['number of sensors'])} & {_d_fmt(_r['hours of operation'])}"
                  f" & {_d_fmt(_r['span (days)'])} \\\\")
_num_cols = ['number of cases', 'number of activities', 'number of sensors',
             'hours of operation', 'span (days)']
_tot = d_datasets[_num_cols].sum(skipna=True)
_lines += [r'\midrule',
           'total & ' + ' & '.join(_d_fmt(_tot[c]) for c in _num_cols) + r' \\']
_lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
print('\n'.join(_lines))


run: experiment_1_20260804_161749


,dataset,number of cases,number of activities,number of sensors,hours of operation,span (days)
0,process_1,300,25,6,604,50
1,process_2,48,12,5,512,21
2,process_3,49,12,5,512,21
3,process_4_1,160,5,18,2387,212
4,process_4_2,156,5,17,2463,205
5,process_5,53,20,28,501,29


totals: 766 cases, 79 sensors, 6979 hours

\begin{table}[width=.9\linewidth,cols=6,pos=h]
\caption{Dataset information overview. Hours of operation refer to the recorded length of the energy time series; span is the calendar period covered.}
\label{table:datasets}
\setlength{\tabcolsep}{3pt}
\begin{tabular}{@{}lrrrrr@{}}
\toprule
dataset & \makecell{number\\of cases} & \makecell{number of\\different\\activities} & \makecell{number\\of sensors} & \makecell{hours of\\operation} & \makecell{span\\(days)} \\
\midrule
process\_1 & 300 & 25 & 6 & 604 & 50 \\
process\_2 & 48 & 12 & 5 & 512 & 21 \\
process\_3 & 49 & 12 & 5 & 512 & 21 \\
process\_4.1 & 160 & 5 & 18 & 2387 & 212 \\
process\_4.2 & 156 & 5 & 17 & 2463 & 205 \\
process\_5 & 53 & 20 & 28 & 501 & 29 \\
\midrule
total & 766 & 79 & 79 & 6979 & 538 \\
\bottomrule
\end{tabular}
\end{table}


<a id="sec-0N"></a>

### 0.N · Aggregated numbers at hand

The counts quoted in the paper, surfaced up front so they can be checked
without scrolling. Two sources:

- **split counts** — read directly from this run's parquets (always current);
- **table populations** (curves, cells, units) — read from the
  `visuals/individual_results_eval*.csv` exports of section 5, i.e. from the
  last **full** notebook run. After switching runs, rerun to the end once
  before trusting those.


In [3]:
# ── N: aggregated numbers at hand ────────────────────────────────────────────
_n_run = _latest_run(EXPERIMENT)
_n_e1 = (pd.read_parquet(_n_run / 'process_eval_per_case.parquet')['split']
           .astype(str).str.upper().value_counts())
_n_e2 = (pd.read_parquet(_n_run / 'curve_eval_results.parquet')['Split']
           .astype(str).str.upper().value_counts())

print(f'run: {_n_run.name}\n')
print('Individual results per evaluation (run parquets — table:eval_counts):')
print(f"  1 Process models      : {_n_e1.get('TRAIN', 0):>8,} train | "
      f"{_n_e1.get('TEST', 0):>8,} test | {int(_n_e1.sum()):>8,} total")
print(f"  2 Individual profiles : {_n_e2.get('TRAIN', 0):>8,} train | "
      f"{_n_e2.get('TEST', 0):>8,} test | {int(_n_e2.sum()):>8,} total")

_n_csv = {i: Path('visuals') / f'individual_results_eval{i}_{n}.csv'
          for i, n in [(1, 'process_timing'), (2, 'individual_profiles'),
                       (3, 'complete_profiles')]}
if all(f.exists() for f in _n_csv.values()):
    _n1, _n2, _n3 = (pd.read_csv(_n_csv[i]) for i in (1, 2, 3))
    _n3_vals = int(_n3['values'].sum())
    print(f"  3 Complete profiles   : {'--':>8} train | {_n3_vals:>8,} test | "
          f"{_n3_vals:>8,} total   (test-only by construction)")
    print(f'  grand total           : '
          f'{int(_n_e1.sum()) + int(_n_e2.sum()) + _n3_vals:,} individual results')

    print('\nTable populations (visuals/individual_results_eval*.csv — last full run):')
    _n1_m = _n1.groupby(['Process method', 'Time approach']).ngroups
    print(f'  1: {len(_n1):,} per-case rows = {_n1_m} (method, time approach) '
          f'combos x {len(_n1) // _n1_m} pooled test cases')
    _n2_a = _n2['Approach'].nunique()
    _n2_c = _n2.groupby(['Activity', 'Sensor', 'Approach']).ngroups
    print(f'  2: {len(_n2):,} curves scored = {_n2_a} approaches x '
          f'{len(_n2) // _n2_a:,} curves -> {_n2_c:,} cells '
          f'({_n2_c // _n2_a} (process, activity, sensor) cells per approach)')
    _n3_u = {m: set(zip(g['sensor'], g['case_id'])) for m, g in _n3.groupby('method')}
    print(f'  3: {len(_n3):,} (method, case, sensor) units over '
          f'{len(_n3_u)} methods -> {_n3_vals:,} metric values (x 5 features) | '
          f'shared across all methods: {len(set.intersection(*_n3_u.values())):,} units')
else:
    print('\n(table populations skipped — visuals/individual_results_eval*.csv '
          'not found; run the notebook to the end once)')


run: experiment_1_20260804_161749

Individual results per evaluation (run parquets — table:eval_counts):
  1 Process models      :    6,420 train |    2,772 test |    9,192 total
  2 Individual profiles :  272,325 train |  135,473 test |  407,798 total
  3 Complete profiles   :       -- train |   63,785 test |   63,785 total   (test-only by construction)
  grand total           : 480,775 individual results

Table populations (visuals/individual_results_eval*.csv — last full run):
  1: 2,079 per-case rows = 9 (method, time approach) combos x 231 pooled test cases
  2: 116,070 curves scored = 6 approaches x 19,345 curves -> 2,340 cells (390 (process, activity, sensor) cells per approach)
  3: 12,757 (method, case, sensor) units over 5 methods -> 63,785 metric values (x 5 features) | shared across all methods: 2,319 units


<a id="sec-0P"></a>

### 0.P · Process discovery — load

In [4]:
# ── P: load process_eval_results, define the metric columns ──────────────────
p_run = _latest_run(EXPERIMENT)
print('P run:', p_run.name)
pdf = pd.read_parquet(p_run / 'process_eval_results.parquet')
_p_sp = P_SPLIT.lower() + '_'
assert any(c.startswith(_p_sp) for c in pdf.columns), f'No {_p_sp}* columns found'
pdf['model'], pdf['time_pred'] = zip(*pdf['mode'].map(_parse_mode))
pdf = pdf.copy()

# Discovery quality FIRST (higher = better), then timing errors (lower = better).
# Every metric carries its direction so highlighting and LaTeX bolding pick the
# right end. Columns: (parquet base, label, LaTeX header, decimals, direction).
_p_dur  = ('duration_metrics_activity_duration_wape'
           if (_p_sp + 'duration_metrics_activity_duration_wape') in pdf.columns
           else 'duration_metrics_activity_duration_error')
_p_span = ('duration_metrics_case_span_wape'
           if (_p_sp + 'duration_metrics_case_span_wape') in pdf.columns
           else 'duration_metrics_case_span_error')

P_SHOW_EVENT_RATIO = True
P_METRICS = [
    ('conformance_metrics_fitness',            'Fitness',        r'Fitness',                      3, 'max'),
    ('conformance_metrics_precision',          'Precision',      r'Precision',                    3, 'max'),
    ('conformance_metrics_generalization',     'Generalization', r'\makecell{General-\\ization}', 3, 'max'),
    ('conformance_metrics_simplicity',         'Simplicity',     r'Simplicity',                   3, 'max'),
]
if P_SHOW_EVENT_RATIO:
    P_METRICS.append(
        ('basic_metrics_event_count_error',    'Evt-Ratio Err',  r'\makecell{Evt-Ratio\\Err}',    3, 'min'))
P_METRICS += [
    (_p_dur,                                   'Activity duration WAPE (%)',   r'\makecell{Activity\\duration\\WAPE (\%)}',    1, 'min'),
    ('duration_metrics_activity_duration_mae', 'Activity duration MAE (min)',  r'\makecell{Dur MAE\\(min)}',    1, 'min'),
    (_p_span,                                  'Lead time WAPE (%)',   r'\makecell{Lead time\\WAPE (\%)}',    1, 'min'),
    ('duration_metrics_case_span_mae',         'Lead time MAE (min)', r'\makecell{Lead time\\MAE (min)}',   1, 'min'),
]
# Dropped: 'Activity duration MAE (min)' is an unweighted mean of absolute minute errors over
# the activity types a case has in COMMON between real and simulated, and each
# method is scored on its OWN intersection. A looser net fires more of the short
# activities, and a short activity can only contribute a small absolute error, so
# the mean is dragged down by composition rather than accuracy -- Alpha "wins" the
# column while losing every scale-free one. Restricting both to the same
# activities reverses it (Alpha 2.04 vs best-net 1.64 min). 'Activity duration WAPE (%)'
# measures the same quantity normalised by the real durations, so it is immune and
# is what the table reports instead.
# 'Lead time MAE (min)' goes for the same reason: case spans run 134 -> 1364 min across
# processes, so pooling absolute minutes mixes incomparable scales (Alpha wins it in
# 4 of 6 processes yet loses pooled). Its standardised form is 'Lead time WAPE (%)', which
# the table keeps. Every error column that survives is now a normalised error,
# median over equally-weighted cases.
P_EXCLUDE_METRICS = ['Activity duration MAE (min)', 'Lead time MAE (min)']

P_METRICS   = [m for m in P_METRICS
               if (_p_sp + m[0]) in pdf.columns and m[1] not in P_EXCLUDE_METRICS]
P_COL_OF    = {l: _p_sp + b for (b, l, t, d, dr) in P_METRICS}
P_LABELS    = [l  for (b, l, t, d, dr) in P_METRICS]
P_TEX_HDR   = {l: t  for (b, l, t, d, dr) in P_METRICS}
P_DECIMALS  = {l: d  for (b, l, t, d, dr) in P_METRICS}
P_DIRECTION = {l: dr for (b, l, t, d, dr) in P_METRICS}


def p_best_of(series, lbl):
    """Direction-aware best of a column slice; None when nothing is finite."""
    v = series.dropna()
    if v.empty:
        return None
    return v.max() if P_DIRECTION[lbl] == 'max' else v.min()


def p_fmt(v, lbl):
    return '' if pd.isna(v) else f'{v:.{P_DECIMALS[lbl]}f}'


print(f'{len(pdf)} rows | processes: {sorted(pdf["process"].unique())}')
print('Discovery:', [l for l in P_LABELS if P_DIRECTION[l] == 'max'])
print('Timing   :', [l for l in P_LABELS if P_DIRECTION[l] == 'min'])

P run: experiment_1_20260804_161749
78 rows | processes: ['process_1', 'process_2', 'process_3', 'process_4_1', 'process_4_2', 'process_5']
Discovery: ['Fitness', 'Precision', 'Generalization', 'Simplicity']
Timing   : ['Evt-Ratio Err', 'Activity duration WAPE (%)', 'Lead time WAPE (%)']


In [5]:
# ── P: resolve the methods, then assemble BOTH aggregation levels ────────────
p_best_miner, _p_cands = _select_best_miner(pdf)


def p_model_for(proc, pm):
    if pm == 'Alpha':         return 'alpha'
    if pm == 'Budget':        return 'budget'
    if pm == 'Combined-best': return p_best_miner.get(proc)
    return None


P_PM_ORDER   = ['Alpha', 'Combined-best', 'Budget']
P_TIME_ORDER = ['baseline', 'ml_global', 'ml_local']
_P_TIME_SUFFIX = {'baseline': '', 'ml_global': '_ml_plus_global',
                  'ml_local': '_ml_plus_per_act'}

# ── Level 1: DISCOVERY quality — one value per process ───────────────────────
# Fitness / Precision / Generalization / Simplicity are MODEL-level: the real log
# replayed against the discovered net, per station, averaged. They have no
# per-case meaning, so they can only be aggregated across processes.
_recs = []
for proc, g in pdf.groupby('process'):
    for pm in P_PM_ORDER:
        mdl = p_model_for(proc, pm)
        if mdl is None:
            continue
        for _, row in g[g['model'] == mdl].iterrows():
            if row['time_pred'] not in P_TIME_ORDER:
                continue
            rec = {'process': proc, 'Process method': pm, 'Time approach': row['time_pred']}
            rec.update({l: row[c] for l, c in P_COL_OF.items()})
            _recs.append(rec)
p_long = pd.DataFrame(_recs)

# ── Level 2: TIMING — one row per CASE ───────────────────────────────────────
# process_eval_per_case.parquet holds the raw per-case sample the pipeline's
# per-process medians were computed from: one row per (process, mode, split,
# case_id). Pooling it directly makes every CASE one unit of the median, instead
# of every process. The two are not the same: case counts are very uneven, so a
# process with many cases now carries proportionally more weight.
P_TIMING_UNIT = 'case'        # 'case' pools per_case rows; 'process' medians the
                              # pipeline's per-process values as before
p_cases = pd.read_parquet(p_run / 'process_eval_per_case.parquet')
p_cases = p_cases[p_cases['split'].astype(str).str.upper() == P_SPLIT.upper()]

_rows = []
for pm in P_PM_ORDER:
    for tp in P_TIME_ORDER:
        for proc in sorted(pdf['process'].unique()):
            mdl = p_model_for(proc, pm)
            if mdl is None:
                continue
            mode = f'petri_net_{mdl}{_P_TIME_SUFFIX[tp]}'
            g = p_cases[(p_cases['process'] == proc) & (p_cases['mode'] == mode)]
            if g.empty:
                print(f'  WARNING: no per-case rows for {proc} / {pm} / {tp} ({mode})')
                continue
            g = g.copy()
            g['Process method'], g['Time approach'] = pm, tp
            _rows.append(g)
p_case_long = pd.concat(_rows, ignore_index=True)

# Which per-case column each reported timing metric comes from. EVERY column is
# the MEDIAN over cases, so every case counts exactly once.
#
# The span column is a pooled WAPE over all test cases: sum(case_span_mae) /
# sum(span_real) * 100. Note it therefore weights each case by its real lead
# time (process_4_1 + 4_2 carry most of the minutes); the column is named
# 'Lead time WAPE (%)'. 'median_pct' = median of a ratio, as a percentage;
# 'pooled_wape' = ratio of sums over the group.
P_PER_CASE = {
    'basic_metrics_event_count_error':          ('evt_ratio_err',  'median'),
    'duration_metrics_activity_duration_wape':  ('dur_wape',       'median'),
    'duration_metrics_activity_duration_error': ('dur_err_activ',  'median'),
    'duration_metrics_activity_duration_mae':   ('dur_mae',        'median'),
    'duration_metrics_case_span_wape':          ('case_span_mae',  'pooled_wape'),
    'duration_metrics_case_span_error':         ('case_span_mae',  'pooled_wape'),
    'duration_metrics_case_span_mae':           ('case_span_mae',  'median'),
}
P_DISC_LABELS = [l for l in P_LABELS if P_DIRECTION[l] == 'max']
P_TIME_LABELS = [l for l in P_LABELS if P_DIRECTION[l] == 'min']
_p_base_of = {l: b for (b, l, t, d, dr) in P_METRICS}
_p_unmapped = [l for l in P_TIME_LABELS if _p_base_of[l] not in P_PER_CASE]
assert not _p_unmapped, f'no per-case column mapped for {_p_unmapped}'

print(f'p_long (discovery): {len(p_long)} rows | {p_long["process"].nunique()} processes')
# case_id repeats across processes, so the timing unit is the (process, case)
# PAIR — counting case_id alone would undercount it.
P_N_CASES = len(p_case_long[['process', 'case_id']].drop_duplicates())
print(f'p_case_long (timing): {len(p_case_long):,} rows | '
      f'{P_N_CASES:,} distinct (process, case) units')
print('cases per (method, time approach):')
print(p_case_long.groupby(['Process method', 'Time approach'])['case_id'].size().to_string())

Combined-best by pipeline, on TRAIN | candidates: ['heuristic', 'inductive']
  process_1: heuristic
  process_2: heuristic
  process_3: heuristic
  process_4_1: heuristic
  process_4_2: heuristic
  process_5: heuristic
p_long (discovery): 54 rows | 6 processes
p_case_long (timing): 2,079 rows | 231 distinct (process, case) units
cases per (method, time approach):
Process method  Time approach
Alpha           baseline         231
                ml_global        231
                ml_local         231
Budget          baseline         231
                ml_global        231
                ml_local         231
Combined-best   baseline         231
                ml_global        231
                ml_local         231


<a id="sec-0I"></a>

### 0.I · Individual profile realism — load

In [6]:
# ── I: config ────────────────────────────────────────────────────────────────
# Raw approach names, dropped everywhere in this evaluation.
I_EXCLUDE_APPROACHES = ['Exemplar (real curve)', 'Exemplar (no DTW)',
                        'Exemplar + DTW (warped)',
                        # 'Baseline' is the per-SENSOR median level ("Sensor median"),
                        # pooled over all activities and objects. It belongs to the
                        # complete-profile evaluation (1.C), where the naive floor is
                        # the comparison. This evaluation's floor is the finer
                        # 'median_activity_sensor', conditioned on exactly the
                        # (sensor, activity, object) leaf the curve models are.
                        'Baseline']
I_MIN_CURVE_POINTS   = 5      # curve-length floor
I_LEVEL_CELL_COVERAGE = True  # restrict to the cells every approach is scored on

# err = |f(pred) - f(real)| / mean|f(real)| per (process, sensor): target 0 everywhere.
I_REALISM_SPECS = {
    'Sum':       ('sum_pred',   'sum_real',   0.0),   # total energy over the curve
    'Max':       ('max_pred',   'max_real',   0.0),   # peak — sizing and tariffs
    'Mean':      ('mean_pred',  'mean_real',  0.0),   # average load level = the bias
    'Std':       ('std_pred',   'std_real',   0.0),   # spread — the flatness measure
    'Roughness': ('rough_pred', 'rough_real', 0.0),   # jaggedness — catches amplitude
                                                      # reached by noise, not dynamics
}
I_REALISM_TARGET = {k: v[2] for k, v in I_REALISM_SPECS.items()}

# Paper naming: the proposed method is 'Step DTW'; every other row is named by what
# it IS, not by what was removed -- the ablation logic belongs in the caption.
I_METHOD_RENAME = {
    'Step DTW smooth + ML + Ext.':    'ML Step DTW (proposed)',
    'Step DTW + ML + Ext.':           'Step DTW (step gains)',
    'ML + Ext. Factors':              'ML DTW (no steps)',
    'DTW + ML + Ext. Factors':        'ML DTW (no steps)',
    'ML DTW':                         'Step DTW w/o segments and ext.',
    'DTW + ML':                       'Step DTW w/o segments and ext.',
    'ML only (no DTW)':               'ML (no DTW)',
    # The naive floor of THIS evaluation, marked as such. 1.C's floor is the
    # coarser per-sensor median and carries the same '(baseline)' marker there.
    'Median per Activity & Sensor':   'Median per Activity & Sensor (baseline)',
    'Median per activity and sensor': 'Median per Activity & Sensor (baseline)',
    'Baseline':                       'Sensor median',
    'DTW + Seq2Seq + Ext. Factors':   'Seq2Seq DTW (aligned)',
    'Seq2Seq IOM (DTW-selected)':     'Seq2Seq (DTW-scored)',
    'DTW + Seq2Seq':                  'Seq2Seq (aligned, w/o ext.)',
    'Seq2Seq only (no DTW)':          'Seq2Seq (unaligned)',
    'Exemplar (real curve)':          'Exemplar (real curve)',
}

i_run = _latest_run(EXPERIMENT)
print('I run:', i_run.name)
i_raw = pd.read_parquet(i_run / 'curve_eval_results.parquet')
i_raw = i_raw[i_raw['Split'] == I_SPLIT]
if I_EXCLUDE_APPROACHES:                       # exclusion uses the RAW names
    i_raw = i_raw[~i_raw['Approach'].isin(I_EXCLUDE_APPROACHES)]

_i_real_path = i_run / 'real_test_curves.parquet'
if 'y_pred' in i_raw.columns and 'y_true' not in i_raw.columns and _i_real_path.exists():
    _i_real = pd.read_parquet(_i_real_path)
    _i_key  = [c for c in ['Process', 'Sensor', 'Activity', 'Instance', 'Split']
               if c in i_raw.columns and c in _i_real.columns]
    i_raw = (i_raw.merge(_i_real[_i_key + ['y_real']], on=_i_key, how='left')
                  .rename(columns={'y_real': 'y_true'}))
    print(f'Joined {len(_i_real):,} real curves on {" + ".join(_i_key)} '
          f'({i_raw["y_true"].notna().mean()*100:.1f}% matched)')

# Anonymise BEFORE any grouping. Sensors are keyed on (process, sensor), so a name
# shared by two processes still gets two distinct labels.
I_ANON_MAPS = {}
if ANONYMISE:
    _i_pair = i_raw['Process'].astype(str) + ' | ' + i_raw['Sensor'].astype(str)
    for _i, _col in enumerate(['Process', 'Activity']):
        if _col in i_raw.columns:
            I_ANON_MAPS[_col] = _anon_map(i_raw[_col], _col, ANON_SEED + _i)
            i_raw[_col] = i_raw[_col].astype(str).map(I_ANON_MAPS[_col])
    I_ANON_MAPS['Sensor'] = _anon_map(_i_pair, 'Sensor', ANON_SEED + 2)
    i_raw['Sensor'] = _i_pair.map(I_ANON_MAPS['Sensor'])
    print('Anonymised: ' + ', '.join(f'{c} ({len(m)})' for c, m in I_ANON_MAPS.items()))

I run: experiment_1_20260804_161749
Joined 58,478 real curves on Process + Sensor + Activity + Instance + Split (100.0% matched)
Anonymised: Process (6), Activity (26), Sensor (79)


In [7]:
# ── I: realism errors, population levelling, cell-level aggregation ──────────
I_CELL_KEY = ['Process', 'Activity', 'Sensor']
_i_before  = i_raw.groupby('Approach').size()

for _side in ('real', 'pred'):
    if f'mean_{_side}' in i_raw.columns and 'N' in i_raw.columns:
        i_raw[f'sum_{_side}'] = (pd.to_numeric(i_raw[f'mean_{_side}'], errors='coerce')
                                 * pd.to_numeric(i_raw['N'], errors='coerce'))

_i_specs = {k: v for k, v in I_REALISM_SPECS.items()
            if v[0] in i_raw.columns and v[1] in i_raw.columns}
_i_missing = sorted({c for s in I_REALISM_SPECS.values() for c in s[:2]} - set(i_raw.columns))

if _i_specs:
    for _name, (_num, _den, _tgt) in _i_specs.items():
        p = pd.to_numeric(i_raw[_num], errors='coerce')
        r = pd.to_numeric(i_raw[_den], errors='coerce')
        # Scale = mean |real| of that (process, sensor), over DISTINCT curves.
        _uniq  = i_raw.drop_duplicates(['Process', 'Sensor', 'Instance'])
        _by_cell = (pd.to_numeric(_uniq[_den], errors='coerce').abs()
                      .groupby([_uniq['Process'], _uniq['Sensor']]).mean())
        _scale = pd.Series(pd.MultiIndex.from_arrays([i_raw['Process'], i_raw['Sensor']])
                             .map(_by_cell).to_numpy(dtype=float), index=i_raw.index)
        i_raw[_name] = np.where(np.isfinite(_scale) & (_scale > 1e-6),
                                (p - r).abs() / _scale, np.nan)
    I_ACTIVE = [m for m in I_REALISM_SPECS if m in _i_specs and i_raw[m].notna().any()]
    print('Realism metrics:', I_ACTIVE)
    if _i_missing:
        print(f'  (absent from this run, skipped: {_i_missing})')
else:
    I_ACTIVE = []
    print('No realism metrics in this run — re-run the pipeline to write the '
          'per-curve shape statistics into curve_eval_results.parquet.')

# Levelling 1 — curve length. Levelling 2 — cell coverage, applied AFTER the
# length floor, because dropping short curves can itself empty a cell for one
# approach and not another.
if I_MIN_CURVE_POINTS:
    i_raw = i_raw[i_raw['N'] >= I_MIN_CURVE_POINTS]
if I_LEVEL_CELL_COVERAGE:
    _cells_of = {a: set(map(tuple, g[I_CELL_KEY].drop_duplicates().to_numpy()))
                 for a, g in i_raw.groupby('Approach')}
    _shared = set.intersection(*_cells_of.values()) if _cells_of else set()
    i_raw = i_raw[i_raw[I_CELL_KEY].apply(tuple, axis=1).isin(_shared)]
    print(f'Cell coverage levelled to the {len(_shared)} (process, activity, sensor) '
          f'cells every approach is scored on')
    for _a, _n in sorted({a: len(c - _shared) for a, c in _cells_of.items() if c - _shared}.items()):
        print(f'  {_a}: dropped {_n} cells no other approach has')

i_levelling = pd.DataFrame({'curves_raw': _i_before,
                            'curves_used': i_raw.groupby('Approach').size()}).fillna(0).astype(int)
i_levelling['dropped'] = i_levelling['curves_raw'] - i_levelling['curves_used']
if i_levelling['curves_used'].nunique() > 1:
    print('curve counts still differ after levelling — not like-for-like.')

i_raw['Approach'] = i_raw['Approach'].replace(I_METHOD_RENAME)   # display names from here

# Stage 1: one value per (process, activity, sensor, approach) = median over its
# curves. Median at BOTH stages — these errors are bounded below with a long right
# tail, so a few bad cells would drag any mean, and a mean of medians is neither.
i_combo = (i_raw.groupby(['Process', 'Activity', 'Sensor', 'Approach'])[I_ACTIVE]
                .median().reset_index())
print(f'{len(i_raw):,} curves -> {len(i_combo):,} (process, activity, sensor, approach) cells | '
      f'{i_raw["Approach"].nunique()} approaches')

Realism metrics: ['Sum', 'Max', 'Mean', 'Std', 'Roughness']
Cell coverage levelled to the 390 (process, activity, sensor) cells every approach is scored on
116,070 curves -> 2,340 (process, activity, sensor, approach) cells | 6 approaches


In [8]:
# ── I: scorecard + table helpers ─────────────────────────────────────────────
def i_realism_deviation(values, metric):
    """Distance to target. Every realism metric is a normalised ABSOLUTE error with
    target 0, so this is the value itself — kept as a function so a signed metric
    added later cannot silently make the ranking bold the worst cell."""
    v = pd.to_numeric(pd.Series(values), errors='coerce')
    return (v - I_REALISM_TARGET.get(metric, 0.0)).abs()


def i_realism_scorecard(df_combo, how='median', metrics=None):
    """Stage 2: collapse the cells into one row per approach, plus 'Overall' = the
    plain average across the metric columns. Rows best-first."""
    metrics = list(metrics if metrics is not None else I_ACTIVE)
    t = df_combo.groupby('Approach')[metrics].agg(how)
    if len(t.columns):
        t['Overall'] = t.mean(axis=1)
        t = t.sort_values('Overall')
    return t


def i_style_realism(t, how='median'):
    def _best(col):
        d = i_realism_deviation(col, col.name)
        return ['font-weight:700;background-color:#d6ecff;'
                if (pd.notna(v) and pd.notna(d.min()) and abs(v - d.min()) < 1e-12)
                else '' for v in d]
    return (t.style.format({m: '{:.3f}' for m in t.columns}, na_rep='—')
             .apply(_best, axis=0)
             .set_caption(f'{I_SPLIT} — {how} over process x activity x sensor; '
                          f'target 0, lower = better'))

<a id="sec-0C"></a>

### 0.C · Complete energy profile — load

In [9]:
# ── C: config ────────────────────────────────────────────────────────────────
# Rows of every C table (order = row order).
C_METHODS = {
    'Baseline':          True,   # per-SENSOR median curve (naive curve generator)
    'Alpha':             True,   # alpha-miner net     + C_CURVE_APPROACH
    'Combined-best':     True,   # best discovered net + C_CURVE_APPROACH
    'Budget':            True,   # best net + duration budgeting + C_CURVE_APPROACH
    'Profile-generator': True,   # stochastic profile generator
}
# Columns of every C table.
C_FEATURE_TOGGLE = {'total': True, 'peak': True, 'mean': True,
                    'std': True, 'roughness': True}

C_CURVE_APPROACH          = 'ml_step_dtw_smooth'
C_BASELINE_CURVE_APPROACH = 'baseline'
C_BASELINE_PROCESS_TYPE   = 'Budget'   # Baseline uses this net; only its CURVES are
                                       # the naive median, so the difference is
                                       # attributable to the curve generator alone.
C_DURATION_MODE = 'ml_local'           # fixed for every process ('best_by_mae' picks
                                       # per process on SELECTION_SPLIT instead)
C_DURATION_SELECT_METRIC = 'duration_metrics_activity_duration_mae'

C_SCHEDULE_SERIES = {'stochastic': 'Profile-generator'}
C_PROCESS_TYPES = {k: C_METHODS.get(k, False) for k in ('Alpha', 'Combined-best', 'Budget')}
C_EXTRA_METHODS = {k: C_METHODS.get(k, False)
                   for k in ('Profile-generator',)}
C_SHOW_BASELINE = C_METHODS.get('Baseline', False)

C_ALL_FEATURES = ['total', 'peak', 'mean', 'std', 'roughness']
C_FEATURES     = [f for f in C_ALL_FEATURES if C_FEATURE_TOGGLE.get(f)]
C_FEAT_LABEL   = {'total': 'Sum', 'peak': 'Max', 'mean': 'Mean',
                  'std': 'Std', 'roughness': 'Roughness'}


# 'total' is the plain sum of the samples: it treats every sample as one minute
# wide, so it carries a small grid-dependent offset (real dt = 1.00 min,
# petri-net ~1.04, stochastic ~1.21) on top of the prediction error.
def c_curve_features(v):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if v.size < 4:
        return None
    return {'total':     float(np.nansum(v)),
            'peak':      float(np.nanmax(v)),
            'mean':      float(np.nanmean(v)),
            'std':       float(np.nanstd(v)),
            'roughness': float(np.nanmean(np.abs(np.diff(v))))}


def c_features_long(df_curves, method_label):
    rows = []
    for (sen, cid), g in df_curves.groupby(['sensor', 'case_id']):
        f = c_curve_features(g.sort_values('t_minutes')['value'].to_numpy())
        if f:
            f.update(sensor=sen, case_id=cid, series=method_label); rows.append(f)
    return rows


c_run = _latest_run(EXPERIMENT)
print('C run:', c_run.name)

C run: experiment_1_20260804_161749


In [10]:
# ── C: resolve process type -> concrete simulation mode per process ──────────
c_pe = pd.read_parquet(c_run / 'process_eval_results.parquet')
c_pe['model'], c_pe['time_pred'] = zip(*c_pe['mode'].map(_parse_mode))
c_best_miner, _c_cands = _select_best_miner(c_pe)
assert _c_cands, 'no miner candidates left after excluding alpha/budget/combined'

_C_TIME_SUFFIX = {'baseline': '', 'ml_global': '_ml_plus_global',
                  'ml_local': '_ml_plus_per_act'}
# C_DURATION_MODE='best_by_mae' picks the variant per process by activity-duration
# MAE. A SELECTION, so it reads the TRAIN column. MAE not WAPE: the comparison is
# within one process, so the scale is constant and MAE stays in minutes.
_c_dur_col = SELECTION_SPLIT.lower() + '_' + C_DURATION_SELECT_METRIC
assert _c_dur_col in c_pe.columns, f'{_c_dur_col} missing'


def c_model_for(proc, ptype):
    return {'Alpha': 'alpha', 'Budget': 'budget'}.get(ptype) or c_best_miner.get(proc)


def c_mode_for(proc, ptype):
    model = c_model_for(proc, ptype)
    if model is None:
        return None
    if C_DURATION_MODE in _C_TIME_SUFFIX:
        tp = C_DURATION_MODE
    else:
        sub = c_pe[(c_pe['process'] == proc) & (c_pe['model'] == model)]
        tp = (sub.loc[sub[_c_dur_col].idxmin(), 'time_pred']
              if not sub.empty and sub[_c_dur_col].notna().any() else 'baseline')
    return f'petri_net_{model}{_C_TIME_SUFFIX[tp]}'


def _c_suffix(approach):   # the pipeline writes the 'baseline' approach unsuffixed
    return '' if approach in ('baseline', '', None) else f'_{approach}'


_c_curve_sfx    = _c_suffix(C_CURVE_APPROACH)
_c_baseline_sfx = _c_suffix(C_BASELINE_CURVE_APPROACH)

Combined-best by pipeline, on TRAIN | candidates: ['heuristic', 'inductive']
  process_1: heuristic
  process_2: heuristic
  process_3: heuristic
  process_4_1: heuristic
  process_4_2: heuristic
  process_5: heuristic


In [11]:
# ── C: build the feature table (real + every method) ─────────────────────────
_c_ptypes = [t for t in ['Alpha', 'Combined-best', 'Budget'] if C_PROCESS_TYPES.get(t)]
_c_extra  = [s for s, lbl in C_SCHEDULE_SERIES.items() if C_EXTRA_METHODS.get(lbl)]
C_METHOD_ORDER = ((['Baseline'] if C_SHOW_BASELINE else []) + _c_ptypes
                  + [C_SCHEDULE_SERIES[s] for s in _c_extra])

_rows = []
for proc in sorted(c_pe['process'].unique()):
    for ptype in _c_ptypes:
        mode = c_mode_for(proc, ptype)
        fp = c_run/'complete_curve_eval_results'/proc/(mode or '')/f'predicted_curves{_c_curve_sfx}.parquet'
        # Warned, not skipped quietly: a missing mode empties the method's row,
        # which is indistinguishable from a method that was switched off.
        if not mode or not fp.exists():
            print(f'  WARNING: {ptype} curves missing for {proc} ({mode}) — row will be empty')
            continue
        for r in c_features_long(pd.read_parquet(fp).query("series == 'predicted'"), ptype):
            r['process'] = proc; _rows.append(r)
    if C_SHOW_BASELINE:
        bmode = c_mode_for(proc, C_BASELINE_PROCESS_TYPE)
        bfp = c_run/'complete_curve_eval_results'/proc/(bmode or '')/f'predicted_curves{_c_baseline_sfx}.parquet'
        if bmode and bfp.exists():
            for r in c_features_long(pd.read_parquet(bfp).query("series == 'predicted'"), 'Baseline'):
                r['process'] = proc; _rows.append(r)
        else:
            print(f'  WARNING: Baseline curves missing for {proc} ({bmode}) — row will be empty')
    sfp = c_run / 'schedule_profile_eval_results' / proc / 'predicted_curves.parquet'
    if not sfp.exists():
        continue
    sdf = pd.read_parquet(sfp)
    for r in c_features_long(sdf[sdf['series'] == 'real'], 'real'):
        r['process'] = proc; _rows.append(r)
    for s in _c_extra:
        for r in c_features_long(sdf[sdf['series'] == s], C_SCHEDULE_SERIES[s]):
            r['process'] = proc; _rows.append(r)
c_feat = pd.DataFrame(_rows)

# Restrict to the sensors common to real + every method within each process.
_keep = []
for proc, g in c_feat.groupby('process'):
    sets = [set(g[g.series == m]['sensor'].unique())
            for m in (['real'] + C_METHOD_ORDER) if m in set(g.series)]
    common = set.intersection(*sets) if sets else set()
    _keep.append(g[g['sensor'].isin(common)])
c_feat = pd.concat(_keep, ignore_index=True)
print('c_feat rows:', len(c_feat), '| methods:', C_METHOD_ORDER)
_c_missing = [m for m in C_METHOD_ORDER if m not in set(c_feat['series'])]
if _c_missing:
    print(f'WARNING: NO DATA for: {_c_missing} — absent from every table')

c_feat rows: 15368 | methods: ['Baseline', 'Alpha', 'Combined-best', 'Budget', 'Profile-generator']


In [12]:
# ── C: one row per (process, case, sensor, method, feature) ──────────────────
# AGGREGATION UNIT = (process, case, sensor). Each case is compared to its OWN
# real counterpart, so a method cannot match the marginal distribution while being
# wrong on every case. Value = |f(pred) - f(real)| / mean|f(real)|, normalised per
# sensor so magnitudes stay comparable across sensors.
_recs = []
for (proc, sen), g in c_feat.groupby(['process', 'sensor']):
    real_g = g[g.series == 'real'].drop_duplicates('case_id').set_index('case_id')
    for feature in C_FEATURES:
        r = real_g[feature].dropna()
        if len(r) < 2:
            continue
        scale = np.nanmean(np.abs(r.to_numpy())) + 1e-9
        # Degenerate-scale guard: for zero-inflated sensors mean|real| collapses
        # and the relative error explodes. Skip those cells.
        if scale < 1e-6:
            continue
        for m in C_METHOD_ORDER:
            q = (g[g.series == m].drop_duplicates('case_id')
                  .set_index('case_id')[feature].dropna())
            shared = r.index.intersection(q.index)
            if len(shared) == 0:
                continue
            for cid, v in ((q.loc[shared] - r.loc[shared]).abs() / scale).items():
                _recs.append({'process': proc, 'sensor': sen, 'case_id': cid,
                              'method': m, 'feature': feature, 'rel_err': float(v)})
c_units = pd.DataFrame(_recs)

# Anonymised HERE: the file paths above are built from the real process names, and
# every table below is produced after this point. A different seed per column, else
# two columns with the same number of labels get the same permutation. Sensor and
# case are keyed on the (process, name) PAIR — sensor names and case ids recur
# across processes, so a name-keyed map would give two different units one label
# and silently merge them once the process column is dropped in 4.C.
C_ANON_MAPS = {}
if ANONYMISE:
    _c_sen_pair  = c_units['process'].astype(str) + ' | ' + c_units['sensor'].astype(str)
    _c_case_pair = c_units['process'].astype(str) + ' | ' + c_units['case_id'].astype(str)
    C_ANON_MAPS['process'] = _anon_map(c_units['process'], 'Process', ANON_SEED + 0)
    C_ANON_MAPS['sensor']  = _anon_map(_c_sen_pair,  'Sensor', ANON_SEED + 1)
    C_ANON_MAPS['case_id'] = _anon_map(_c_case_pair, 'Case',   ANON_SEED + 2)
    c_units['process'] = c_units['process'].astype(str).map(C_ANON_MAPS['process'])
    c_units['sensor']  = _c_sen_pair.map(C_ANON_MAPS['sensor'])
    c_units['case_id'] = _c_case_pair.map(C_ANON_MAPS['case_id'])
    print('Anonymised: ' + ', '.join(f'{c} ({len(m)})' for c, m in C_ANON_MAPS.items()))

print(f'c_units: {len(c_units):,} rows | '
      f'{c_units[["process","case_id","sensor"]].drop_duplicates().shape[0]:,} '
      f'distinct (process, case, sensor)')

Anonymised: process (6), sensor (75), case_id (231)
c_units: 63,785 rows | 2,611 distinct (process, case, sensor)


In [13]:
# ── C: scorecard + row descriptions ──────────────────────────────────────────
def c_scorecard(df_units):
    t = (df_units.groupby(['method', 'feature'])['rel_err'].median().unstack('feature')
                 .reindex(index=C_METHOD_ORDER, columns=C_FEATURES))
    t.columns = [C_FEAT_LABEL[c] for c in t.columns]
    t['Overall'] = t.mean(axis=1)     # simple average across the metric columns
    t = t.sort_values('Overall')      # best first
    t.index.name = 'Method'
    return t


# The duration predictor is part of the process model: C_DURATION_MODE picks it per
# process, so it is summarised as the majority choice with (n/processes) appended
# when the processes did not all pick the same one.
_C_TIME_LABEL = {'baseline': 'baseline duration', 'ml_global': 'ML_global',
                 'ml_local': 'ml_local'}


def _c_time_variant(ptype):
    picks = []
    for proc in sorted(c_pe['process'].unique()):
        mode = c_mode_for(proc, ptype)
        if not mode:
            continue
        picks.append('ml_global' if mode.endswith('_ml_plus_global') else
                     'ml_local'  if mode.endswith('_ml_plus_per_act') else 'baseline')
    if not picks:
        return ''
    cnt = pd.Series(picks).value_counts()
    return _C_TIME_LABEL[cnt.index[0]] + ('' if len(cnt) == 1 else f' ({cnt.iloc[0]}/{len(picks)})')


# One label per row: the method AND the process model it is generated from.
C_METHOD_LABEL = {
    'Baseline':          'Median per Sensor (baseline), no process model',
    'Alpha':             f'Alpha Petri net + {_c_time_variant("Alpha")}',
    'Combined-best':     f'Best Petri net + {_c_time_variant("Combined-best")}',
    'Budget':            f'Best Petri net + Budget + {_c_time_variant("Budget")}',
    'Profile-generator': 'Profile-generator (no process model)',
}
C_METHOD_CURVE_GEN = {
    'Baseline':          'Median per Sensor',
    'Alpha':             'Step DTW',
    'Combined-best':     'Step DTW',
    'Budget':            'Step DTW',
    'Profile-generator': 'Stochastic generator',
}


def c_with_description(tbl):
    t = tbl.copy()
    t.insert(0, 'Curve generation', [C_METHOD_CURVE_GEN.get(m, '') for m in t.index])
    t.index = [C_METHOD_LABEL.get(m, m) for m in t.index]
    t.index.name = 'Method (process model)'
    return t


def c_style_score(tbl):
    t = c_with_description(tbl)
    num = [c for c in t.columns if c != 'Curve generation']
    return (t.style.format('{:.3f}', subset=num, na_rep='—')
              .highlight_min(axis=0, subset=num,
                             props='font-weight:700;background-color:#d6ecff;')
              .set_caption('Median per-case error to real over (process, case, sensor) '
                           '— lower = closer to real'))

---
<a id="sec-1"></a>

# 1 · Results

The three headline tables. Everything they are made of is in section 4.

Realism / profile columns are all `err = |f(pred) − f(real)| / mean|f(real)|`,
median over units — **0 = perfect, lower = better**; `Overall` is their average.
`Sum` total energy · `Max` peak load · `Mean` bias · `Std` amplitude of the
dynamics · `Roughness` `mean|Δv|`, jaggedness (the two together separate real
dynamics from added noise).

<a id="sec-1P"></a>

## 1.P · Process discovery + timing

In [14]:
# ── P: the headline table — two aggregation levels side by side ─────────────
# Discovery columns: median across PROCESSES (model-level, no per-case meaning).
# Timing columns:    pooled across CASES (see P_TIMING_UNIT).
def p_timing_pooled(g):
    # One pooled timing value per metric, from the per-case rows of `g`.
    out = {}
    for lbl in P_TIME_LABELS:
        col, how = P_PER_CASE[_p_base_of[lbl]]
        if how == 'pooled_wape':
            out[lbl] = float(g['case_span_mae'].sum() / g['span_real'].sum() * 100)
            continue
        v = pd.to_numeric(g[col], errors='coerce').replace([np.inf, -np.inf], np.nan)
        out[lbl] = float(v.median()) * (100 if how == 'median_pct' else 1)
    return pd.Series(out)


def p_build_combined(agg='median'):
    idx = pd.MultiIndex.from_tuples(
        [(pm, tp) for pm in P_PM_ORDER for tp in P_TIME_ORDER],
        names=['Process method', 'Time approach'])
    disc = (p_long.groupby(['Process method', 'Time approach'])[P_DISC_LABELS].agg(agg)
                  .reindex(idx))
    # Per-case fitness: new pipeline runs export a 'fitness' column in
    # process_eval_per_case.parquet (real trace replayed against the discovered
    # net, one value per case). Pool it over cases — median over all (process,
    # case) units — like the timing metrics. Precision / Generalization /
    # Simplicity stay process-level: they have no per-case form. Runs without
    # the column keep the process-level fitness median.
    if P_FITNESS_PER_CASE and 'Fitness' in disc.columns:
        disc['Fitness'] = (p_case_long.groupby(['Process method', 'Time approach'])['fitness']
                                      .median().reindex(idx))
    if P_TIMING_UNIT == 'case':
        time = (p_case_long.groupby(['Process method', 'Time approach'])
                           .apply(p_timing_pooled).reindex(idx))
    else:
        time = (p_long.groupby(['Process method', 'Time approach'])[P_TIME_LABELS].agg(agg)
                      .reindex(idx))
    return pd.concat([disc, time], axis=1)[P_LABELS].dropna(how='all')


P_FITNESS_PER_CASE = ('fitness' in p_case_long.columns
                      and p_case_long['fitness'].notna().any())

p_combined = p_build_combined(agg=P_AGG)


def p_style(tbl, best, caption):
    def _styler(_):
        out = pd.DataFrame('', index=tbl.index, columns=tbl.columns)
        for idx in tbl.index:
            for col in tbl.columns:
                v, b = tbl.loc[idx, col], best.get(col)
                if b is not None and pd.notna(v) and abs(v - b) < 1e-9:
                    out.loc[idx, col] = 'font-weight:700;background-color:#d6ecff;'
        return out
    fmt = {l: (lambda v, ll=l: p_fmt(v, ll)) for l in tbl.columns}
    return tbl.style.format(fmt).apply(_styler, axis=None).set_caption(caption)


_p_unit = (f'pooled over the {P_N_CASES:,} test cases'
           if P_TIMING_UNIT == 'case' else f'{P_AGG} across processes')
_p_fit_unit = ('median over pooled test cases' if P_FITNESS_PER_CASE
               else f'{P_AGG} across processes')
display(p_style(p_combined, {c: p_best_of(p_combined[c], c) for c in p_combined.columns},
                f'Discovery quality (higher = better), fitness {_p_fit_unit}, others {P_AGG} across processes | '
                f'timing errors (lower = better), {_p_unit} — {P_SPLIT} set'))

<a id="sec-1I"></a>

## 1.I · Individual profile realism

In [15]:
if not I_ACTIVE:
    display(Markdown('> **Skipped — no realism metrics in this run.**'))
else:
    i_realism_median = i_realism_scorecard(i_combo, how='median')
    display(i_style_realism(i_realism_median))

,Sum,Max,Mean,Std,Roughness,Overall
Approach,,,,,,
ML Step DTW (proposed),0.015,0.064,0.044,0.354,0.470,0.189
ML DTW (no steps),0.015,0.073,0.047,0.431,0.540,0.221
ML (no DTW),0.015,0.079,0.048,0.446,0.556,0.229
Seq2Seq (DTW-scored),0.018,0.085,0.061,0.528,0.646,0.267
Seq2Seq DTW (aligned),0.016,0.081,0.057,0.528,0.665,0.270
Median per Activity & Sensor (baseline),0.012,0.078,0.041,0.536,0.712,0.276


<a id="sec-1C"></a>

## 1.C · Complete energy profile

In [16]:
c_score_all = c_scorecard(c_units)
display(c_style_score(c_score_all))

,Curve generation,Sum,Max,Mean,Std,Roughness,Overall
Method (process model),,,,,,,
Best Petri net + Budget + ml_local,Step DTW,0.211,0.074,0.077,0.372,0.602,0.267
Best Petri net + ml_local,Step DTW,0.379,0.077,0.070,0.385,0.605,0.303
Alpha Petri net + ml_local,Step DTW,0.431,0.083,0.076,0.447,0.625,0.332
"Median per Sensor (baseline), no process model",Median per Sensor,0.320,0.165,0.088,0.908,0.942,0.485
Profile-generator (no process model),Stochastic generator,0.348,0.197,0.092,1.280,9.980,2.379


---
<a id="sec-2"></a>

# 2 · LaTeX for the paper

Preamble: `booktabs`, `caption`, `float`, `array`, `tabularx`, `makecell`,
`multirow`. With `SAVE_LATEX` each table is also written under `visuals/`.

<a id="sec-2P"></a>

## 2.P · Process table

In [17]:
# ── P: LaTeX layout knobs ────────────────────────────────────────────────────
P_LATEX_TABCOLSEP = '8pt'              # column padding (LaTeX default 6pt)
P_PAPER_STRETCH   = '1.3'
P_PAPER_FONT      = r'\footnotesize'

P_PAPER_ROW = {                        # level-0 row labels of the paper table
    'Alpha':         r'\makecell[c]{Alpha \\ Petri net \\ (Baseline)}',
    'Combined-best': r'\makecell[c]{Best \\ Petri net}',
    'Budget':        r'\makecell[c]{Best \\ Petri net \\ + Budget}',
}
P_PAPER_HDR = {
    'Fitness':        r'Fitness',
    'Precision':      r'Precision',
    'Generalization': r'\makecell{Generalization}',
    'Simplicity':     r'Simplicity',
    'Evt-Ratio Err':  r'\makecell{Evt-Ratio\\Error}',
    'Activity duration WAPE (%)':   r'\makecell{Activity\\duration\\WAPE (\%)}',
    'Activity duration MAE (min)':  r'\makecell{Activity\\duration\\MAE (min)}',
    'Lead time WAPE (%)':   r'\makecell{Lead time\\WAPE (\%)}',
    'Lead time MAE (min)': r'\makecell{Case \\span MAE\\(min)}',
}
# Decimals used ONLY by the paper table: substring rules first, then exact-label
# overrides, then P_DECIMALS.
P_PAPER_DECIMALS_RULES = [('WAPE', 3)]
P_PAPER_DECIMALS       = {}

P_TEX_IDX_NAME = {'Process method': 'Method',
                  'Time approach': r'\makecell[l]{Time\\approach}'}

def _p_paper_dp(lbl):
    if lbl in P_PAPER_DECIMALS:
        return P_PAPER_DECIMALS[lbl]
    for pat, dp in P_PAPER_DECIMALS_RULES:
        if pat.lower() in str(lbl).lower():
            return dp
    return P_DECIMALS[lbl]


def p_to_paper_table(tbl, caption, label, description):
    """`table*` float in the paper layout: caption on top, booktabs rules, one
    \\multirow block per process method separated by \\cline, description under it.
    Bolding is global per column and direction-aware."""
    n_idx = tbl.index.nlevels
    ncol  = n_idx + len(tbl.columns)
    best  = {c: p_best_of(tbl[c], c) for c in tbl.columns}

    def cell(v, col):
        if pd.isna(v):
            return ''
        s, b = f'{v:.{_p_paper_dp(col)}f}', best[col]
        return (r'\textbf{' + s + '}') if (b is not None and abs(v - b) < 1e-9) else s

    hdr = ([P_TEX_IDX_NAME.get(n, str(n)) for n in tbl.index.names]
           + [P_PAPER_HDR.get(c, P_TEX_HDR[c]) for c in tbl.columns])
    L = [r'\begin{table*}[t]', r'\centering',
         rf'\caption{{{caption}}}', rf'\label{{{label}}}', r'\vspace{-0.5em}',
         rf'\setlength{{\tabcolsep}}{{{P_LATEX_TABCOLSEP}}}',
         rf'\renewcommand{{\arraystretch}}{{{P_PAPER_STRETCH}}}', P_PAPER_FONT,
         # No vertical rules — journal house style (booktabs \toprule/\midrule/\bottomrule only).
         r'\begin{tabular}{' + 'l' * n_idx + 'c' * len(tbl.columns) + '}',
         r'\toprule', ' & '.join(hdr) + r' \\', r'\midrule']
    for pm in dict.fromkeys(tbl.index.get_level_values(0)):      # keeps P_PM_ORDER
        sub = tbl.xs(pm, level=0, drop_level=False)
        head = rf'\multirow{{{len(sub)}}}{{*}}{{{P_PAPER_ROW.get(pm, _tex_esc(pm))}}}'
        for idx, row in sub.iterrows():
            L.append(' & '.join([head, _tex_esc(idx[-1])]
                                + [cell(row[c], c) for c in tbl.columns]) + r' \\')
            head = ''
        L.append(rf'\cline{{1-{ncol}}}')
    L += [r'\bottomrule', r'\end{tabular}', r'\vspace{0.5em}',
          r'\noindent\raggedright\footnotesize', '', description, '', r'\end{table*}']
    return '\n'.join(L)

In [18]:
# ── P: the paper table ───────────────────────────────────────────────────────
P_PAPER_DESC = (
    r'Process models: Alpha Petri net (baseline miner), Best Petri net (best discovered net) and '
    r'Best Petri net + Budget (adds duration budgeting at simulation time), each '
    r'crossed with three activity-duration predictors: baseline samples the fitted '
    r'statistical distributions, ml\_global is one ML algorithm for all activities, '
    r'ml\_local one per activity. Fitness, Precision, Generalization and Simplicity '
    r'score discovery quality (higher is better; Fitness is the median over all '
    r'pooled test cases when per-case fitness is available, the others the median '
    r'across processes, as they are model-level metrics with no per-case form). Evt-Ratio '
    r'Error is the simulated-to-real event-count ratio error (lower is better). '
    r'Activity duration and Lead time are WAPEs against the real test cases (lower is '
    r'better): the former the median over cases, the latter pooled over them. '
    r'\textbf{Bold} = best per column.')

p_tex_agg = p_to_paper_table(
    p_combined,
    caption=(f'Process modeling and timing accuracy ({P_SPLIT} set results)'),
    label='tab:process_results',
    description=P_PAPER_DESC)
print(p_tex_agg)

if SAVE_LATEX:
    Path('visuals').mkdir(exist_ok=True)
    _f = Path('visuals') / 'process_results.tex'
    _f.write_text('% Process discovery + timing — \\input{} this file.\n' + p_tex_agg + '\n')
    print('\nSaved:', _f)

\begin{table*}[t]
\centering
\caption{Process modeling and timing accuracy (test set results)}
\label{tab:process_results}
\vspace{-0.5em}
\setlength{\tabcolsep}{8pt}
\renewcommand{\arraystretch}{1.3}
\footnotesize
\begin{tabular}{llccccccc}
\toprule
Method & \makecell[l]{Time\\approach} & Fitness & Precision & \makecell{Generalization} & Simplicity & \makecell{Evt-Ratio\\Error} & \makecell{Activity\\duration\\WAPE (\%)} & \makecell{Lead time\\WAPE (\%)} \\
\midrule
\multirow{3}{*}{\makecell[c]{Alpha \\ Petri net \\ (Baseline)}} & baseline & 0.827 & 0.400 & 0.586 & 0.489 & 0.333 & 39.440 & 68.311 \\
 & ml\_global & 0.827 & 0.400 & 0.586 & 0.489 & 0.333 & 27.375 & 57.677 \\
 & ml\_local & 0.827 & 0.400 & 0.586 & 0.489 & 0.333 & 25.795 & 58.033 \\
\cline{1-9}
\multirow{3}{*}{\makecell[c]{Best \\ Petri net}} & baseline & \textbf{1.000} & \textbf{0.728} & \textbf{0.664} & \textbf{0.660} & 0.333 & 43.875 & 74.272 \\
 & ml\_global & \textbf{1.000} & \textbf{0.728} & \textbf{0.664} & \textbf{

<a id="sec-2I"></a>

## 2.I · Individual profile realism table

In [19]:
# ── I: paper float (table* + minipage + caption + note) ──────────────────────
I_PAPER_HDR      = {'Roughness': 'Roughness'}
I_PAPER_MINIPAGE = '16cm'     # minipage holding caption + tabular
I_PAPER_FIRSTCOL = '6cm'      # p{} width of the method column
I_PAPER_NOTEBOX  = '16cm'     # parbox width of the note under the table
I_PAPER_DECIMALS = 3


def _i_cells(t, dp=3):
    """Formatted strings with the per-column best (closest to target) bolded."""
    s = pd.DataFrame(index=t.index, columns=t.columns, dtype=object)
    for col in t.columns:
        vals = t[col].dropna()
        d    = i_realism_deviation(vals, col).dropna()
        best = vals.get(d.idxmin()) if not d.empty else None
        for idx in t.index:
            v = t.loc[idx, col]
            s.loc[idx, col] = ('' if pd.isna(v) else
                               (r'\textbf{' + f'{v:.{dp}f}' + '}')
                               if (best is not None and abs(v - best) < 1e-6)
                               else f'{v:.{dp}f}')
    return s


def i_to_latex_paper(t, caption, label, note):
    s = _i_cells(t, I_PAPER_DECIMALS)
    body = '\n'.join(' & '.join([_tex_esc(I_METHOD_RENAME.get(idx, idx))]
                                + [s.loc[idx, c] for c in t.columns]) + r' \\'
                     for idx in t.index)
    # Headers are plain text — no \textit / \textbf, matching the P and C tables.
    hdr = ' & '.join(['Method']
                     + [I_PAPER_HDR.get(c, _tex_esc(c)) for c in t.columns])
    return '\n'.join([
        r'\begin{table*}[H]', r'\centering', '',
        rf'\begin{{minipage}}{{{I_PAPER_MINIPAGE}}}', r'\centering', '',
        r'\captionsetup{', r'    justification=centering,',
        r'    singlelinecheck=false,', r'    format=plain', r'}', '',
        rf'\caption{{{caption}}}', rf'\label{{{label}}}', '',
        r'\vspace{-0.5em}', '',
        # No vertical rules — journal house style.
        rf'\begin{{tabular}}{{p{{{I_PAPER_FIRSTCOL}}}' + 'c' * len(t.columns) + '}',
        r'\toprule', hdr + r' \\', r'\midrule', body, r'\bottomrule', r'\end{tabular}', '',
        r'\vspace{0.5em}', '',
        rf'\parbox{{{I_PAPER_NOTEBOX}}}{{%', r'\footnotesize', note, r'}', '',
        r'\end{minipage}', '', r'\end{table*}'])


I_REALISM_NOTE = (
    'Curve realism for individual profile prediction, {split} set. Each column is the '
    'median normalised absolute error of that curve property, so 0 is perfect and '
    'lower is better; Overall is their average.\n'
    r'\textbf{{Bold}} marks the best value per column.').format(split=I_SPLIT)

if not I_ACTIVE:
    print('No realism metrics in this run — nothing to emit.')
else:
    i_tex_agg = i_to_latex_paper(
        i_realism_median,
        caption='Curve realism for individual energy-profile prediction.',
        label='tab:individual_profile_realism',
        note=I_REALISM_NOTE)
    print(i_tex_agg)
    if SAVE_LATEX:
        Path('visuals').mkdir(exist_ok=True)
        _f = Path('visuals') / 'individual_profile_realism.tex'
        _f.write_text('% Aggregated realism table — \\input{} this file.\n' + i_tex_agg + '\n')
        print('\nSaved:', _f)

\begin{table*}[H]
\centering

\begin{minipage}{16cm}
\centering

\captionsetup{
    justification=centering,
    singlelinecheck=false,
    format=plain
}

\caption{Curve realism for individual energy-profile prediction.}
\label{tab:individual_profile_realism}

\vspace{-0.5em}

\begin{tabular}{p{6cm}cccccc}
\toprule
Method & Sum & Max & Mean & Std & Roughness & Overall \\
\midrule
ML Step DTW (proposed) & 0.015 & \textbf{0.064} & 0.044 & \textbf{0.354} & \textbf{0.470} & \textbf{0.189} \\
ML DTW (no steps) & 0.015 & 0.073 & 0.047 & 0.431 & 0.540 & 0.221 \\
ML (no DTW) & 0.015 & 0.079 & 0.048 & 0.446 & 0.556 & 0.229 \\
Seq2Seq (DTW-scored) & 0.018 & 0.085 & 0.061 & 0.528 & 0.646 & 0.267 \\
Seq2Seq DTW (aligned) & 0.016 & 0.081 & 0.057 & 0.528 & 0.665 & 0.270 \\
Median per Activity \& Sensor (baseline) & \textbf{0.012} & 0.078 & \textbf{0.041} & 0.536 & 0.712 & 0.276 \\
\bottomrule
\end{tabular}

\vspace{0.5em}

\parbox{16cm}{%
\footnotesize
Curve realism for individual profile predictio

<a id="sec-2C"></a>

## 2.C · Complete energy profile table

In [20]:
# ── C: table* + minipage, caption on top, method note at the bottom ──────────
# Typeset with tabularx, so it always fits the text width and the long descriptions
# wrap by themselves — no manual cm widths, no resizebox.
C_LATEX_MINIPAGE   = r'\textwidth'
C_LATEX_NOTE_WIDTH = r'\linewidth'
C_LATEX_FONT       = r'\small'
C_LATEX_TEXT_W     = (1.30, 0.70)   # relative widths of Method / Curve generation; sum = 2
C_GROUP_BY_TYPE    = True           # row order + rules by type; the type itself is not
                                    # printed — the two text columns already say it
C_METHOD_TYPE = {
    'Baseline':          'Baseline',
    'Alpha':             'Process model',
    'Combined-best':     'Process model',
    'Budget':            'Process model',
    'Profile-generator': 'Schedule-based',
}
C_TYPE_ORDER = ['Process model', 'Schedule-based', 'Baseline']

C_METHOD_NOTE = (
    r'\textit{Median per Sensor (baseline)}: one median level per sensor, pooled over all '
    r'its activities, emitted as a flat line for every case. '
    r'\textit{Alpha Petri net}: net discovered by the alpha miner. '
    r'\textit{Best Petri net}: best discovered net per process, selected on the '
    r'training split by the mean of Fitness, Precision, Generalization and Simplicity. '
    r'\textit{Best Petri net + Budget}: the same net, with each case generated to match '
    r'its predicted total-duration budget. '
    r'\textit{Profile-generator}: stochastic profile generator. '
    r'The three Petri-net rows use the \textit{Step DTW} curve predictor '
    r'of Evaluation~2. '
    r'Each row names the process model the cases are generated from, including its '
    r'duration predictor (ml\_local = one per activity), and \textit{Curve generation} '
    r'is how the load curve of each activity or case is then produced.')

C_METRIC_NOTE = (r'Cells are the median over (process, case, sensor) of the paired per-case '
                 r'relative error $|f(\mathrm{pred})-f(\mathrm{real})|/\overline{|f(\mathrm{real})|}$. '
                 r'Lower is better; \textbf{bold} = best per column. '
                 r'Overall is the average across the metric columns.')


def _c_row_order(tbl):
    """Rows grouped by type (best Overall first inside each group), or flat."""
    if not C_GROUP_BY_TYPE:
        return list(tbl.index)
    key = 'Overall' if 'Overall' in tbl.columns else tbl.columns[0]
    order = []
    for typ in C_TYPE_ORDER:
        grp = [m for m in tbl.index if C_METHOD_TYPE.get(m) == typ]
        order += sorted(grp, key=lambda m: (pd.isna(tbl.loc[m, key]), tbl.loc[m, key]))
    return order + [m for m in tbl.index if m not in order]


def c_to_latex_score(tbl, caption, label, note_extra=''):
    cols = list(tbl.columns)
    best = {c: tbl[c].dropna().min() for c in cols if tbl[c].notna().any()}

    def cell(m, c):
        v = tbl.loc[m, c]
        if pd.isna(v):
            return '--'
        s = f'{v:.3f}'
        return r'\textbf{' + s + '}' if abs(v - best.get(c, np.inf)) < 1e-9 else s

    order = _c_row_order(tbl)
    _x = [(r'>{\hsize=' + f'{w:g}' + r'\hsize\linewidth=\hsize'
           r'\raggedright\arraybackslash}X') for w in C_LATEX_TEXT_W]
    # No vertical rules — journal house style.
    colfmt = ''.join(_x) + 'c' * len(cols)
    # Headers are plain text — no \textbf / \textit, matching the P and I tables.
    head = ('Method (process model) & Curve generation & '
            + ' & '.join(str(c) for c in cols) + r' \\')
    body, i = [], 0
    while i < len(order):
        typ, span = C_METHOD_TYPE.get(order[i], ''), 1
        if C_GROUP_BY_TYPE:
            while i + span < len(order) and C_METHOD_TYPE.get(order[i + span]) == typ:
                span += 1
        for k in range(span):
            mm = order[i + k]
            body.append(' & '.join([_tex_esc(C_METHOD_LABEL.get(mm, mm)),
                                    _tex_esc(C_METHOD_CURVE_GEN.get(mm, ''))]
                                   + [cell(mm, c) for c in cols]) + r' \\')
        if C_GROUP_BY_TYPE and i + span < len(order):
            body.append(r'\midrule')
        i += span
    return '\n'.join([
        r'\begin{table*}[H]', r'\centering', '',
        f'\\begin{{minipage}}{{{C_LATEX_MINIPAGE}}}', r'\centering', '',
        r'\captionsetup{', r'    justification=centering,',
        r'    singlelinecheck=false,', r'    format=plain', r'}', '',
        r'\caption{' + caption + '}', r'\label{' + label + '}', '',
        r'\vspace{-0.5em}', '', C_LATEX_FONT,
        f'\\begin{{tabularx}}{{\\linewidth}}{{{colfmt}}}', r'\toprule', head, r'\midrule',
        *body, r'\bottomrule', r'\end{tabularx}', '', r'\vspace{0.5em}', '',
        f'\\parbox{{{C_LATEX_NOTE_WIDTH}}}{{%', r'\footnotesize',
        C_METHOD_NOTE + (' ' + note_extra if note_extra else ''),
        '}', '', r'\end{minipage}', '', r'\end{table*}'])


c_tex_all = c_to_latex_score(
    c_score_all,
    caption='Complete energy-profile comparison, all processes.',
    label='tab:energy_profile',
    note_extra=C_METRIC_NOTE)
print(c_tex_all)

if SAVE_LATEX:
    Path('visuals').mkdir(exist_ok=True)
    _f = Path('visuals') / 'complete_energy_profile.tex'
    _f.write_text('% Complete energy-profile table — \\input{} this file.\n' + c_tex_all + '\n')
    print('\nSaved:', _f)

\begin{table*}[H]
\centering

\begin{minipage}{\textwidth}
\centering

\captionsetup{
    justification=centering,
    singlelinecheck=false,
    format=plain
}

\caption{Complete energy-profile comparison, all processes.}
\label{tab:energy_profile}

\vspace{-0.5em}

\small
\begin{tabularx}{\linewidth}{>{\hsize=1.3\hsize\linewidth=\hsize\raggedright\arraybackslash}X>{\hsize=0.7\hsize\linewidth=\hsize\raggedright\arraybackslash}Xcccccc}
\toprule
Method (process model) & Curve generation & Sum & Max & Mean & Std & Roughness & Overall \\
\midrule
Best Petri net + Budget + ml\_local & Step DTW & \textbf{0.211} & \textbf{0.074} & 0.077 & \textbf{0.372} & \textbf{0.602} & \textbf{0.267} \\
Best Petri net + ml\_local & Step DTW & 0.379 & 0.077 & \textbf{0.070} & 0.385 & 0.605 & 0.303 \\
Alpha Petri net + ml\_local & Step DTW & 0.431 & 0.083 & 0.076 & 0.447 & 0.625 & 0.332 \\
\midrule
Profile-generator (no process model) & Stochastic generator & 0.348 & 0.197 & 0.092 & 1.280 & 9.980 & 2.379 \\

---
<a id="sec-3"></a>

# 3 · Evaluation counts

How many evaluations each row above is a median over. The comparisons are only
like-for-like if every method is scored on the **same** population; equal counts
are necessary but not sufficient, so the shared set is intersected explicitly.
Any shortfall is flagged.

<a id="sec-3S"></a>

## 3.S · Summary — individual results per evaluation

One row per evaluation: what a single result is, and how many there are on
the training and test splits. Evaluation 3 is test-only by construction
(only test cases are simulated). These are the numbers quoted in the paper's
evaluation paragraphs.

In [21]:
# ── S: individual-result counts per evaluation ───────────────────────────────
_s_run = _latest_run(EXPERIMENT)
_s_pc = pd.read_parquet(_s_run / 'process_eval_per_case.parquet')
_s_cv = pd.read_parquet(_s_run / 'curve_eval_results.parquet')
_s_e1 = _s_pc['split'].astype(str).str.upper().value_counts()
_s_e2 = _s_cv['Split'].astype(str).str.upper().value_counts()
_s_e3 = len(c_units) if 'c_units' in globals() else None

eval_counts = pd.DataFrame([
    {'evaluation': '1 · Process models',
     'one result per': 'process x method x case',
     'train': int(_s_e1.get('TRAIN', 0)), 'test': int(_s_e1.get('TEST', 0))},
    {'evaluation': '2 · Individual profiles',
     'one result per': 'approach x profile instance',
     'train': int(_s_e2.get('TRAIN', 0)), 'test': int(_s_e2.get('TEST', 0))},
    {'evaluation': '3 · Complete profiles',
     'one result per': 'method x case x sensor x feature',
     'train': 0, 'test': int(_s_e3) if _s_e3 is not None else 0},
]).set_index('evaluation')
eval_counts['total'] = eval_counts['train'] + eval_counts['test']
display(eval_counts)
print(f"grand total: {eval_counts['total'].sum():,} individual results\n")

_s_lines = [
    r'\begin{table}[width=.9\linewidth,cols=5,pos=h]',
    r'\caption{Number of individual evaluation results per split. The reported'
    r' metrics are calculated on the test split; evaluation 3 simulates only'
    r' test cases.}',
    r'\label{table:eval_counts}',
    r'\setlength{\tabcolsep}{3pt}',
    r'\begin{tabular}{@{}llrrr@{}}',
    r'\toprule',
    r'evaluation & one result per & train & test & total \\',
    r'\midrule',
]
def _s_num(v):
    return f'{v:,}'.replace(',', '{,}')
for _name, _r in eval_counts.iterrows():
    _unit = _r['one result per'].replace(' x ', r' $\times$ ')
    _tr = _s_num(_r['train']) if _r['train'] else '--'
    _s_lines.append(f"{_name.split(chr(183))[-1].strip()} & {_unit} & {_tr}"
                    f" & {_s_num(_r['test'])} & {_s_num(_r['total'])} " + r'\\')
_s_lines += [r'\midrule',
             'total & & ' + f"{_s_num(eval_counts['train'].sum())} & "
             f"{_s_num(eval_counts['test'].sum())} & "
             f"{_s_num(eval_counts['total'].sum())} " + r'\\',
             r'\bottomrule', r'\end{tabular}', r'\end{table}']
print('\n'.join(_s_lines))


,one result per,train,test,total
evaluation,,,,
1 · Process models,process x method x case,6420,2772,9192
2 · Individual profiles,approach x profile instance,272325,135473,407798
3 · Complete profiles,method x case x sensor x feature,0,63785,63785


grand total: 480,775 individual results

\begin{table}[width=.9\linewidth,cols=5,pos=h]
\caption{Number of individual evaluation results per split. The reported metrics are calculated on the test split; evaluation 3 simulates only test cases.}
\label{table:eval_counts}
\setlength{\tabcolsep}{3pt}
\begin{tabular}{@{}llrrr@{}}
\toprule
evaluation & one result per & train & test & total \\
\midrule
Process models & process $\times$ method $\times$ case & 6{,}420 & 2{,}772 & 9{,}192 \\
Individual profiles & approach $\times$ profile instance & 272{,}325 & 135{,}473 & 407{,}798 \\
Complete profiles & method $\times$ case $\times$ sensor $\times$ feature & -- & 63{,}785 & 63{,}785 \\
\midrule
total & & 278{,}745 & 202{,}030 & 480{,}775 \\
\bottomrule
\end{tabular}
\end{table}


In [22]:
def _report_uneven(counts, unit, diag=()):
    """Verdict line for a counts table: every non-diagnostic column must be constant."""
    bad = [c for c in counts.columns if c not in diag and counts[c].nunique(dropna=False) > 1]
    if not bad:
        print(f'OK - like-for-like: every row is scored on the same '
              f'{int(counts.iloc[0, 0])} {unit}')
        return bad
    print(f'WARNING: counts differ — the medians are NOT taken over the same {unit}:')
    for c in bad:
        hi = counts[c].max()
        print(f'   {c}: max {hi}, short rows -> '
              + ', '.join(f'{i}={v}' for i, v in counts[c].items() if v != hi))
    return bad

<a id="sec-3P"></a>

## 3.P · Process — processes (discovery) and cases (timing)

In [23]:
# Two populations, because the P table mixes two aggregation levels:
#   processes -- the unit of the four discovery columns
#   cases     -- the unit of every timing column (P_TIMING_UNIT = 'case')
_g = p_long.groupby(['Process method', 'Time approach'])
p_eval_counts = pd.DataFrame({'processes': _g['process'].nunique()})
for _l in P_DISC_LABELS:
    p_eval_counts[_l] = _g[_l].count()
_gc = p_case_long.groupby(['Process method', 'Time approach'])
p_eval_counts['cases'] = _gc['case_id'].size()
for _l in P_TIME_LABELS:
    p_eval_counts[_l] = _gc[P_PER_CASE[_p_base_of[_l]][0]].count()
p_eval_counts = p_eval_counts.reindex(pd.MultiIndex.from_tuples(
    [(pm, tp) for pm in P_PM_ORDER for tp in P_TIME_ORDER
     if (pm, tp) in p_eval_counts.index],
    names=['Process method', 'Time approach'])).fillna(0).astype(int)

display(p_eval_counts)
_report_uneven(p_eval_counts, 'processes / cases')

# Cases per process behind the timing columns. Weighting is NOT uniform: the
# pooled median gives a process with more cases proportionally more influence.
_share = (p_case_long.drop_duplicates(['process', 'case_id'])
                     .groupby('process').size().sort_values(ascending=False))
display(Markdown('**Cases per process** — the weight each process carries in every '
                 'pooled timing number'))
display(pd.DataFrame({'cases': _share,
                      'share of all cases': (_share / _share.sum() * 100).round(1)}))

processes  Fitness  Precision  Generalization  \
Process method Time approach                                                  
Alpha          baseline               6        6          6               6   
               ml_global              6        6          6               6   
               ml_local               6        6          6               6   
Combined-best  baseline               6        6          6               6   
               ml_global              6        6          6               6   
               ml_local               6        6          6               6   
Budget         baseline               6        6          6               6   
               ml_global              6        6          6               6   
               ml_local               6        6          6               6   

                              Simplicity  cases  Evt-Ratio Err  \
Process method Time approach                                     
Alpha          baseline                6    231            231   
               ml_global               6    231            231   
               ml_local                6    231            231   
Combined-best  baseline                6    231            231   
               ml_global               6    231            231   
               ml_local                6    231            231   
Budget         baseline                6    231            231   
               ml_global               6    231            231   
               ml_local                6    231            231   

                              Activity duration WAPE (%)  Lead time WAPE (%)  
Process method Time approach                                                  
Alpha          baseline                              229                 231  
               ml_global                             229                 231  
               ml_local                              229                 231  
Combined-best  baseline                              231                 231  
               ml_global                             231                 231  
               ml_local                              231                 231  
Budget         baseline                              231                 231  
               ml_global                             231                 231  
               ml_local                              231                 231

   Activity duration WAPE (%): max 231, short rows -> ('Alpha', 'baseline')=229, ('Alpha', 'ml_global')=229, ('Alpha', 'ml_local')=229


**Cases per process** — the weight each process carries in every pooled timing number

,cases,share of all cases
process,,
process_1,90,39.0
process_4_1,48,20.8
process_4_2,47,20.3
process_5,16,6.9
process_2,15,6.5
process_3,15,6.5


<a id="sec-3I"></a>

## 3.I · Individual profile — one (process, activity, sensor) cell

In [24]:
# curves = per-curve rows of curve_eval_results; cells = the (process, activity,
# sensor) medians they collapse to, and ONE CELL IS ONE UNIT of every I median.
_KEY  = ['Process', 'Activity', 'Sensor']
_sets = {a: set(map(tuple, g[_KEY].drop_duplicates().to_numpy()))
         for a, g in i_combo.groupby('Approach')}
_common = set.intersection(*_sets.values()) if _sets else set()

i_eval_counts = pd.DataFrame({'curves': i_raw.groupby('Approach').size(),
                              'cells':  i_combo.groupby('Approach').size()})
for _m in I_ACTIVE:
    i_eval_counts[f'{_m} cells'] = i_combo.groupby('Approach')[_m].count()
i_eval_counts['shared cells']   = pd.Series({a: len(s & _common) for a, s in _sets.items()})
i_eval_counts['outside shared'] = pd.Series({a: len(s - _common) for a, s in _sets.items()})
i_eval_counts = i_eval_counts.fillna(0).astype(int).sort_index()

display(Markdown('**Population levelled** — curves with $\\geq$ '
                 f'{I_MIN_CURVE_POINTS} samples'
                 + (', on the cells shared by every approach' if I_LEVEL_CELL_COVERAGE else '')))
display(i_levelling)
display(i_eval_counts)
_report_uneven(i_eval_counts, '(process, activity, sensor) cells',
               diag=('shared cells', 'outside shared'))

**Population levelled** — curves with $\geq$ 5 samples, on the cells shared by every approach

,curves_raw,curves_used,dropped
Approach,,,
DTW + Seq2Seq + Ext. Factors,19345,19345,0
ML + Ext. Factors,19345,19345,0
ML only (no DTW),19345,19345,0
Median per Activity & Sensor,19345,19345,0
Seq2Seq IOM (DTW-selected),19345,19345,0
Step DTW smooth + ML + Ext.,19345,19345,0


,curves,cells,Sum cells,Max cells,Mean cells,Std cells,Roughness cells,shared cells,outside shared
Approach,,,,,,,,,
ML (no DTW),19345,390,390,390,390,390,390,390,0
ML DTW (no steps),19345,390,390,390,390,390,390,390,0
ML Step DTW (proposed),19345,390,390,390,390,390,390,390,0
Median per Activity & Sensor (baseline),19345,390,390,390,390,390,390,390,0
Seq2Seq (DTW-scored),19345,390,390,390,390,390,390,390,0
Seq2Seq DTW (aligned),19345,390,390,390,390,390,390,390,0


OK - like-for-like: every row is scored on the same 19345 (process, activity, sensor) cells


[]

<a id="sec-3C"></a>

## 3.C · Complete profile — one (process, case, sensor) unit

In [25]:
_UKEY = ['process', 'case_id', 'sensor']
_sets = {m: set(map(tuple, g[_UKEY].drop_duplicates().to_numpy()))
         for m, g in c_units.groupby('method')}
_common = set.intersection(*_sets.values()) if _sets else set()

_g = c_units.groupby('method')
c_eval_counts = pd.DataFrame({
    'rows':      _g.size(),
    'units':     _g[_UKEY].apply(lambda d: len(d.drop_duplicates())),
    'processes': _g['process'].nunique(),
    'sensors':   _g.apply(lambda d: len(d[['process', 'sensor']].drop_duplicates())),
    'cases':     _g.apply(lambda d: len(d[['process', 'case_id']].drop_duplicates())),
})
_per_feat = c_units.pivot_table(index='method', columns='feature', values='rel_err',
                                aggfunc='count', fill_value=0)
_per_feat = _per_feat[[f for f in C_FEATURES if f in _per_feat.columns]]
_per_feat.columns = [C_FEAT_LABEL.get(c, c) for c in _per_feat.columns]
c_eval_counts['shared units']   = pd.Series({m: len(s & _common) for m, s in _sets.items()})
c_eval_counts['outside shared'] = pd.Series({m: len(s - _common) for m, s in _sets.items()})
c_eval_counts = (c_eval_counts.join(_per_feat).reindex(C_METHOD_ORDER)
                              .fillna(0).astype(int))
c_eval_counts.index.name = 'Method'

display(c_eval_counts)
_report_uneven(c_eval_counts, '(process, case, sensor) units',
               diag=('shared units', 'outside shared'))

,rows,units,processes,sensors,cases,shared units,outside shared,Sum,Max,Mean,Std,Roughness
Method,,,,,,,,,,,,
Baseline,12305,2461,4,65,201,2319,142,2461,2461,2461,2461,2461
Alpha,12475,2495,6,75,224,2319,176,2495,2495,2495,2495,2495
Combined-best,12895,2579,6,75,230,2319,260,2579,2579,2579,2579,2579
Budget,13055,2611,6,75,231,2319,292,2611,2611,2611,2611,2611
Profile-generator,13055,2611,6,75,231,2319,292,2611,2611,2611,2611,2611


   rows: max 13055, short rows -> Baseline=12305, Alpha=12475, Combined-best=12895
   units: max 2611, short rows -> Baseline=2461, Alpha=2495, Combined-best=2579
   processes: max 6, short rows -> Baseline=4
   sensors: max 75, short rows -> Baseline=65
   cases: max 231, short rows -> Baseline=201, Alpha=224, Combined-best=230
   Sum: max 2611, short rows -> Baseline=2461, Alpha=2495, Combined-best=2579
   Max: max 2611, short rows -> Baseline=2461, Alpha=2495, Combined-best=2579
   Mean: max 2611, short rows -> Baseline=2461, Alpha=2495, Combined-best=2579
   Std: max 2611, short rows -> Baseline=2461, Alpha=2495, Combined-best=2579
   Roughness: max 2611, short rows -> Baseline=2461, Alpha=2495, Combined-best=2579


['rows',
 'units',
 'processes',
 'sensors',
 'cases',
 'Sum',
 'Max',
 'Mean',
 'Std',
 'Roughness']

---
<a id="sec-4"></a>

# 4 · Full tables (anonymised)

What the section-1 tables are made of, down to the **individual result** of
every evaluation: one row per case (**P**), per curve (**I**) and per
(case, sensor) unit (**C**). The coarser levels use the same values and the
same statistic, so the top of each ladder reproduces the headline table
exactly.

**Nothing is keyed on the process** — even a shuffled process number would
still say which sensors and cases belong to the same plant. All remaining
identity labels (activity, sensor, case, curve) are shuffled numbers under
`ANONYMISE`; sensor, case and curve labels are keyed on the *(process, name)*
pair, so a name shared by two processes still gets two distinct labels without
referencing either. The labels group rows correctly but name nothing. Printed
**unrounded** — a zero here is a real zero, not a rounding artefact.

With `SAVE_INDIVIDUAL_CSV` the complete individual-level table of each
evaluation is also written to `visuals/individual_results_eval*.csv`. P and C
are displayed in full; the displayed I table is capped at
`I_INDIVIDUAL_MAX_ROWS` (its ~100k rows of HTML would make this file
unopenable) — its CSV always holds every row.

<a id="sec-4P"></a>

## 4.P · Process — individual per-case results

In [26]:
# ── P: every individual per-case result behind the 1.P timing numbers ────────
# One row per (Process method, Time approach, case): the raw per-case values the
# pooled medians and WAPEs of 1.P are computed from. The case label is a shuffled
# number keyed on the (process, case) pair — no process column, no name, and no
# two processes share a label. 'Lead time WAPE' has no per-case value of its own:
# it is pooled, sum(abs err)/sum(real), from the two lead-time columns here.
SAVE_INDIVIDUAL_CSV = True

_p_ind_cols = ({'Fitness': 'fitness'} if P_FITNESS_PER_CASE else {})
_p_ind_cols.update({'Evt-Ratio Err':              'evt_ratio_err',
                    'Activity duration WAPE (%)': 'dur_wape',
                    'Lead time abs. err (min)':   'case_span_mae',
                    'Real lead time (min)':       'span_real'})
_p_ind_cols = {k: v for k, v in _p_ind_cols.items() if v in p_case_long.columns}

p_ind = p_case_long.copy()
_p_pair = p_ind['process'].astype(str) + ' | ' + p_ind['case_id'].astype(str)
if ANONYMISE:
    # Same seed and key format as C's case map, so a case that appears in both
    # 4.P and 4.C keeps one label — consistent, but still nameless.
    p_ind['Case'] = _p_pair.map(_anon_map(_p_pair, 'Case', ANON_SEED + 2))
else:
    p_ind['Case'] = _p_pair
p_ind = (p_ind[['Process method', 'Time approach', 'Case'] + list(_p_ind_cols.values())]
         .rename(columns={v: k for k, v in _p_ind_cols.items()}))

_p_ncombo = p_ind.groupby(['Process method', 'Time approach']).ngroups
p_ind['_pm'] = pd.Categorical(p_ind['Process method'], P_PM_ORDER, ordered=True)
p_ind['_tp'] = pd.Categorical(p_ind['Time approach'], P_TIME_ORDER, ordered=True)
p_ind['_c']  = pd.to_numeric(p_ind['Case'].astype(str).str.extract(r'(\d+)$', expand=False),
                             errors='coerce')
p_ind = (p_ind.sort_values(['_pm', '_tp', '_c']).drop(columns=['_pm', '_tp', '_c'])
              .set_index(['Process method', 'Time approach', 'Case']))

display(Markdown(f'### Unit — one case · {len(p_ind):,} rows — {P_N_CASES:,} test '
                 f'cases × {_p_ncombo} (process method, time approach) combinations'))
display(HTML(p_ind.to_html()))   # full precision, every row
if SAVE_INDIVIDUAL_CSV:
    Path('visuals').mkdir(exist_ok=True)
    _f = Path('visuals') / 'individual_results_eval1_process_timing.csv'
    p_ind.to_csv(_f); print(f'Saved: {_f} ({len(p_ind):,} rows)')

### Unit — one case · 2,079 rows — 231 test cases × 9 (process method, time approach) combinations

Saved: visuals/individual_results_eval1_process_timing.csv (2,079 rows)


<a id="sec-4I"></a>

## 4.I · Individual profile — individual curves + breakdown ladder

In [27]:
# ── I: every individual per-curve result ─────────────────────────────────────
# One row per (curve, approach): the raw pre-aggregation realism errors that the
# cell medians below — and section 1.I — collapse. 'Curve' is a shuffled number
# keyed on the full (process, activity, sensor, instance) tuple, so the same real
# curve keeps one label across approaches (comparable per curve) and the raw
# instance id, which can carry case names or timestamps, is never printed.
# ~100k rows: the DISPLAY is truncated (head + tail, pandas style) — embedding it
# all as HTML makes the notebook unopenable — but the CSV holds every row.
I_INDIVIDUAL_MAX_ROWS = 1000

if not I_ACTIVE:
    display(Markdown('> **Skipped — no realism metrics in this run.**'))
else:
    i_ind = i_raw.copy()
    _i_curve = (i_ind['Process'].astype(str) + ' | ' + i_ind['Activity'].astype(str) +
                ' | ' + i_ind['Sensor'].astype(str) + ' | ' + i_ind['Instance'].astype(str))
    i_ind['Curve'] = (_i_curve.map(_anon_map(_i_curve, 'Curve', ANON_SEED + 3))
                      if ANONYMISE else _i_curve)
    i_ind['Overall'] = i_ind[I_ACTIVE].mean(axis=1)     # same definition as section 1
    for _col in ('Activity', 'Sensor', 'Curve'):
        i_ind['_' + _col] = pd.to_numeric(
            i_ind[_col].astype(str).str.extract(r'(\d+)$', expand=False), errors='coerce')
    i_ind = (i_ind.sort_values(['_Activity', '_Sensor', '_Curve', 'Approach'])
                  .set_index(['Activity', 'Sensor', 'Curve', 'Approach'])
                  [['N'] + I_ACTIVE + ['Overall']])
    display(Markdown(f'### Individual curves — one (curve, approach) row · '
                     f'{len(i_ind):,} rows, {I_SPLIT} split'
                     + ('' if I_INDIVIDUAL_MAX_ROWS is None or len(i_ind) <= I_INDIVIDUAL_MAX_ROWS
                        else f' — showing {I_INDIVIDUAL_MAX_ROWS}, every row is in the CSV')))
    display(HTML(i_ind.to_html(max_rows=I_INDIVIDUAL_MAX_ROWS)))
    if SAVE_INDIVIDUAL_CSV:
        Path('visuals').mkdir(exist_ok=True)
        _f = Path('visuals') / 'individual_results_eval2_individual_profiles.csv'
        i_ind.to_csv(_f); print(f'Saved: {_f} ({len(i_ind):,} rows)')

### Individual curves — one (curve, approach) row · 116,070 rows, TEST split — showing 1000, every row is in the CSV

Saved: visuals/individual_results_eval2_individual_profiles.csv (116,070 rows)


In [28]:
# Median realism at each level, with the population behind every row. No Process
# column: it is an identifier, not a result, and nothing here is reported per
# process. Nothing merges silently without it — sensor labels are numbered per
# (process, sensor) pair, so a name shared by two processes still gets two rows.
I_LADDER = [('Unit — one (activity, sensor) cell', ['Activity', 'Sensor']),
            ('Activity',                           ['Activity']),
            ('Sensor',                             ['Sensor'])]


def _breakdown_sortkey(col):
    """Sort 'Sensor 12'-style labels by their number; leave real values alone."""
    if col.dtype != object:
        return col
    n = pd.to_numeric(col.astype(str).str.extract(r'(\d+)$', expand=False),
                      errors='coerce')
    return col if n.isna().any() else n


def i_breakdown(keys):
    grp = keys + ['Approach']
    t = i_combo.groupby(grp)[I_ACTIVE].median()
    t['Overall'] = t.mean(axis=1)              # same definition as section 1
    t['cells']   = i_combo.groupby(grp).size()  # (process, activity, sensor) units
    t['curves']  = i_raw.groupby(grp).size()    # per-curve evaluations behind them
    return t.sort_values(keys + ['Overall'], key=_breakdown_sortkey)


if not I_ACTIVE:
    display(Markdown('> **Skipped — no realism metrics in this run.**'))
else:
    # Bare HTML rather than display(df) or a Styler: at ~2,000 rows both roughly
    # double the size of the .ipynb, and there is no single best row to highlight
    # inside a breakdown anyway.
    for title, keys in I_LADDER:
        t = i_breakdown(keys)                   # full precision, not rounded
        display(Markdown(f'### {title} — {t.index.droplevel(-1).nunique()} groups, '
                         f'{len(t):,} rows ({t["curves"].sum():,} curves)'))
        display(HTML(t.to_html()))

### Unit — one (activity, sensor) cell — 390 groups, 2,340 rows (116,070 curves)

### Activity — 26 groups, 156 rows (116,070 curves)

### Sensor — 79 groups, 474 rows (116,070 curves)

<a id="sec-4C"></a>

## 4.C · Complete profile — breakdown ladder

In [29]:
# With ANONYMISE the unit level is keyed on (sensor, case) only: the process column
# is dropped, because even a shuffled 'Process 4' still says which sensors and cases
# belong to the same plant. Still the true (process, sensor, case) unit without it:
# sensor and case labels are numbered per (process, name) pair, so nothing merges.
C_UNIT_KEY = ['sensor', 'case_id'] if ANONYMISE else ['process', 'sensor', 'case_id']
C_LADDER   = [(f'Unit — one ({", ".join(k.replace("_id", "") for k in C_UNIT_KEY)}) cell',
               C_UNIT_KEY),
              ('Sensor', ['sensor'])]

# None = every row: the unit level IS this evaluation's individual-results table
# (~16k rows), shown in full — the saved notebook grows by a few MB for it.
C_BREAKDOWN_MAX_ROWS = None
C_BREAKDOWN_CSV      = False


def c_breakdown(keys):
    grp = keys + ['method']
    t = (c_units.pivot_table(index=grp, columns='feature', values='rel_err',
                             aggfunc='median')
                .reindex(columns=[f for f in C_FEATURES if f in set(c_units['feature'])]))
    t.columns = [C_FEAT_LABEL[c] for c in t.columns]
    t['Overall'] = t.mean(axis=1)                                  # as in the scorecard
    t['units']   = c_units.groupby(grp)[['process', 'case_id', 'sensor']].apply(
                       lambda d: len(d.drop_duplicates()))
    t['values']  = c_units.groupby(grp).size()
    return t.sort_values(keys + ['Overall'], key=_breakdown_sortkey)


for title, keys in C_LADDER:
    t = c_breakdown(keys)
    display(Markdown(f'### {title} — {t.index.droplevel(-1).nunique()} groups, '
                     f'{len(t):,} rows'
                     + ('' if C_BREAKDOWN_MAX_ROWS is None or len(t) <= C_BREAKDOWN_MAX_ROWS
                        else f' (showing {C_BREAKDOWN_MAX_ROWS})')))
    display(HTML(t.to_html(max_rows=C_BREAKDOWN_MAX_ROWS)))
    if keys == C_UNIT_KEY and SAVE_INDIVIDUAL_CSV:
        Path('visuals').mkdir(exist_ok=True)
        _f = Path('visuals') / 'individual_results_eval3_complete_profiles.csv'
        t.to_csv(_f); print(f'Saved: {_f} ({len(t):,} rows)')
    if C_BREAKDOWN_CSV:
        Path('visuals').mkdir(exist_ok=True)
        _f = Path('visuals') / f'complete_profile_breakdown_{"_".join(keys)}.csv'
        t.to_csv(_f); print('Saved:', _f)

### Unit — one (sensor, case) cell — 2611 groups, 12,757 rows

Saved: visuals/individual_results_eval3_complete_profiles.csv (12,757 rows)


### Sensor — 75 groups, 365 rows